# Almond Tree Segmentation

In [ ]:

"""from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("snir5").project("almons-trees")
version = project.version(8)
dataset = version.download("coco-segmentation")
                """
                

### Trsformations functions

In [ ]:
import cv2, numpy as np
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pycocotools.coco import COCO
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm import tqdm
import glob
import random
import matplotlib.pyplot as plt


random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

IMAGE_SIZE = 1024



def gray_world_wb(img):
    # img uint8 RGB
    imgf = img.astype(np.float32)
    mean = imgf.reshape(-1,3).mean(axis=0) + 1e-6
    scale = mean.mean() / mean
    out = np.clip(imgf * scale, 0, 255).astype(np.uint8)
    return out

def adaptive_gamma(img, target_v=0.5, clip=(0.7, 1.4)):
    # move average luminance toward target_v deterministically
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    v = hsv[...,2].astype(np.float32)/255.0
    v_mean = float(np.clip(v.mean(), 0.05, 0.95))
    gamma = np.log(v_mean) / np.log(max(target_v, 1e-6))
    gamma = float(np.clip(gamma, clip[0], clip[1]))
    x = (img.astype(np.float32)/255.0) ** (1.0/gamma)
    return np.clip(x*255.0,0,255).astype(np.uint8)

def adaptive_clahe(img, base_clip=2.0, tile=(8,8)):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    v = hsv[...,2]
    v_std = float(v.std())/255.0
    # lower contrast -> stronger CLAHE; keep bounded
    clip_limit = float(np.clip(base_clip + (0.8 - v_std)*1.0, 1.5, 3.0))
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile)
    hsv[...,2] = clahe.apply(v)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

def adaptive_color_deterministic(img):
    # 1) white balance  2) gamma to target  3) CLAHE by contrast
    wb  = gray_world_wb(img)
    gam = adaptive_gamma(wb, target_v=0.5, clip=(0.8, 1.3))
    out = adaptive_clahe(gam, base_clip=2.0, tile=(8,8))
    return out


### Normalization 

In [ ]:
# Different normalization options
NORM_TECHNIQUES = {
    "imagenet": {
        "mean": (0.485, 0.456, 0.406),
        "std": (0.229, 0.224, 0.225)
    },
    "dataset": None,  # Will compute from dataset
    "minmax": {"mean": (0.0, 0.0, 0.0), "std": (1.0, 1.0, 1.0)},
    "none": None
}

def compute_dataset_mean_std(img_dir, sample_size=500):
    """Compute mean and std for dataset (RGB in [0,1])"""
    import glob
    import cv2
    import numpy as np

    img_paths = glob.glob(os.path.join(img_dir, "*.jpg"))[:sample_size]
    means, stds = [], []

    for path in img_paths:
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) / 255.0
        means.append(img.mean(axis=(0, 1)))
        stds.append(img.std(axis=(0, 1)))

    mean = np.mean(means, axis=0)
    std = np.mean(stds, axis=0)
    return tuple(mean), tuple(std)


def get_base_transform(norm_type="imagenet", dataset_img_dir=None):
    if norm_type == "dataset" and dataset_img_dir:
        mean, std = compute_dataset_mean_std(dataset_img_dir)
        print(f"📊 Dataset normalization: mean={mean}, std={std}")
        norm_transform = A.Normalize(mean=mean, std=std)
    elif norm_type == "imagenet":
        mean_std = NORM_TECHNIQUES["imagenet"]
        norm_transform = A.Normalize(mean=mean_std["mean"], std=mean_std["std"])
    elif norm_type == "minmax":
        norm_transform = A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0))
    elif norm_type == "none":
        norm_transform = A.Lambda(image=lambda x, **kwargs: x)  # no change
    else:
        raise ValueError(f"Unknown normalization type: {norm_type}")

    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Equalize(mode='cv', p=0.5),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.5),
        A.Sharpen(alpha=(0.1, 0.3), lightness=(0.7, 1.0), p=0.4),
        norm_transform,
        ToTensorV2()
    ])


In [ ]:

# ImageNet (default for pretrained models)
train_transform = get_base_transform(norm_type="imagenet")
'''
# Example: Min-Max [0, 1] normalization
train_transform = get_base_transform(norm_type="minmax")

# Example: No normalization
train_transform = get_base_transform(norm_type="none")

# Example: Dataset-based normalization
train_transform = get_base_transform(norm_type="dataset", dataset_img_dir=train_img_dir)
'''

### Datasets loading, Transformation and Training set Augmention

In [ ]:


os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Device setup
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# 📐 Constants

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_val_test_transform_adaptive(IMAGE_SIZE=1024, norm="imagenet"):
    if norm == "imagenet":
        norm_tf = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    elif norm == "minmax":
        norm_tf = A.Normalize(mean=(0,0,0), std=(1,1,1))
    elif norm == "none":
        norm_tf = A.Lambda(image=lambda x, **k: x)
    else:
        raise ValueError("norm must be 'imagenet' | 'minmax' | 'none'")

    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Lambda(image=lambda x, **k: adaptive_color_deterministic(x)),  # deterministic, per-image
        norm_tf,
        ToTensorV2()
    ])

# 🧱 Base transform for *all* datasets — to enhance tree clarity consistently
def get_base_transform():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Equalize(mode='cv', p=0.5),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.5),
        A.Sharpen(alpha=(0.1, 0.3), lightness=(0.7, 1.0), p=0.4),
        A.Normalize(),
        ToTensorV2()
    ])


# 🌱 Full transform for TRAINING dataset: base + tree-specific augmentations
def get_train_transform():
    base = get_base_transform()
    aug = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.3),
        A.Transpose(p=0.3),
        A.RandomGamma(gamma_limit=(60, 140), p=0.4),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.4),
        A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=15, p=0.4),
        A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.3),
        A.Emboss(alpha=(0.2, 0.5), strength=(0.2, 0.6), p=0.3)
    ])
    return A.Compose(aug.transforms + base.transforms)  # Combine augmentations + base


class COCOSegmentationDataset(Dataset):
    def __init__(self, img_dir, ann_path, transform):
        self.img_dir = img_dir
        self.coco = COCO(ann_path)
        self.image_ids = list(self.coco.imgs.keys())
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info['file_name'])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = np.fliplr(image)  # Flip image

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        for ann in anns:
            mask = np.maximum(mask, self.coco.annToMask(ann))
        mask = np.fliplr(mask)  # Flip mask to match image

        if mask.shape[:2] != image.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

        augmented = self.transform(image=image, mask=mask)
        image = augmented['image']
        mask = (augmented['mask'] > 0).unsqueeze(0).float()
        return image, mask




# 🛠️ Set up paths
train_img_dir = "/Users/snirtahasa/Almond_Research/Training/Notebooks/Almons-Trees-8/train"
train_ann_path = os.path.join(train_img_dir, "_annotations.coco.json")
val_img_dir = "/Users/snirtahasa/Almond_Research/Training/Notebooks/Almons-Trees-8/valid"
val_ann_path = os.path.join(val_img_dir, "_annotations.coco.json")

# 📦 Load transforms
# keep your current random train transform
train_transform = get_train_transform()  # your existing function

# deterministic, adaptive color for val & test
val_transform  = get_val_test_transform_adaptive(IMAGE_SIZE=IMAGE_SIZE, norm="imagenet")
test_transform = get_val_test_transform_adaptive(IMAGE_SIZE=IMAGE_SIZE, norm="imagenet")


# 📦 Create datasets and dataloaders
train_dataset = COCOSegmentationDataset(train_img_dir, train_ann_path, train_transform)
val_dataset = COCOSegmentationDataset(val_img_dir, val_ann_path, val_transform)

BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 👁️ Visual preview
def show_batch(images, masks):
    for i in range(len(images)):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        mask = masks[i][0].cpu().numpy()

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(img)
        plt.title("Image")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(mask, cmap='gray')
        plt.title("Mask")
        plt.axis('off')
        plt.show()

# Preview one batch
#for images, masks in train_loader:
   # show_batch(images, masks)
  #  break

# Preview one batch
for images, masks in val_loader:
    show_batch(images, masks)
    break



### Mask Alignment Validation

In [ ]:
def check_mask_alignment(dataset, idx=0, alpha=0.5, hflip=True, vflip=False):
    """
    Visual overlay that mirrors the geometry ops in __getitem__.
    - hflip: apply np.fliplr to image & mask (default True to match your __getitem__)
    - vflip: optional vertical flip if you ever need it
    """
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    # --- load raw image ---
    img_id = dataset.image_ids[idx]
    img_info = dataset.coco.loadImgs(img_id)[0]
    img_path = os.path.join(dataset.img_dir, img_info['file_name'])

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # --- build raw mask from COCO anns ---
    ann_ids = dataset.coco.getAnnIds(imgIds=img_id)
    anns = dataset.coco.loadAnns(ann_ids)
    mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
    for ann in anns:
        mask = np.maximum(mask, dataset.coco.annToMask(ann))

    # --- apply the SAME geometry as in __getitem__ ---
    if hflip:
        image = np.fliplr(image)
        mask  = np.fliplr(mask)
    if vflip:
        image = np.flipud(image)

    # --- size guard (shouldn’t be needed, but safe) ---
    if mask.shape[:2] != image.shape[:2]:
        mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

    # --- overlay ---
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.imshow(mask, cmap=ListedColormap(['none', 'lime']), alpha=alpha)
    plt.axis("off")
    plt.title("Image + Mask Overlay (geometry-matched to __getitem__)")
    plt.show()


In [ ]:
check_mask_alignment(val_dataset)



In [ ]:
def show_exact_raw(idx, dataset):
    img_id = dataset.image_ids[idx]
    img_info = dataset.coco.loadImgs(img_id)[0]
    img_path = os.path.join(dataset.img_dir, img_info['file_name'])

    # Read raw image
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Build raw mask
    ann_ids = dataset.coco.getAnnIds(imgIds=img_id)
    anns = dataset.coco.loadAnns(ann_ids)
    mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)

    for ann in anns:
        mask = np.maximum(mask, dataset.coco.annToMask(ann))

    # Show raw image and raw mask
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='gray')
    plt.title("Original Mask")
    plt.axis('off')
    plt.show()

show_exact_raw(0, val_dataset)


### Model Constraction

In [ ]:
# Updated Model: Use EfficientNet-B3 with no final activation
model_ImNet_Plus = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

### Loss Function 

In [ ]:
import torch
import torch.nn as nn

# Manual implementation of Focal Tversky Loss
class FocalTverskyLoss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, smooth=1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)  # Apply sigmoid here since activation=None
        targets = targets

        TP = (inputs * targets).sum(dim=(1, 2, 3))
        FP = ((1 - targets) * inputs).sum(dim=(1, 2, 3))
        FN = (targets * (1 - inputs)).sum(dim=(1, 2, 3))

        tversky = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        focal_tversky = (1 - tversky) ** self.gamma

        return focal_tversky.mean()


In [ ]:
loss_fn = FocalTverskyLoss(alpha=0.3, beta=0.7, gamma=0.75)

# Optimizer
optimizer = torch.optim.Adam(model_ImNet_Plus.parameters(), lr=1e-3)

# Scheduler with fixed closing parenthesis
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2)

In [ ]:
def dice_coef(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2. * intersection + eps) / (union + eps)
    return dice.mean()

def iou_score(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()


### Model Training

In [ ]:
# Initialize history tracking
history = {
    'train_loss': [],
    'val_loss': [],
    'train_dice': [],
    'val_dice': [],
    'train_iou': [],
    'val_iou': []
}

# Best score tracking
best_val_dice = 0
patience = 5
epochs_no_improve = 0

# Scheduler for dynamic learning rate
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

# Main training loop
for epoch in range(1, 21):
    model_ImNet_Plus.train()
    train_loss = 0
    train_dice = 0
    train_iou = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model_ImNet_Plus(images)
        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_dice += dice_coef(outputs, masks).item()
        train_iou += iou_score(outputs, masks).item()

    model_ImNet_Plus.eval()
    val_loss = 0
    val_dice = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            images, masks = images.to(device), masks.to(device)
            outputs = model_ImNet_Plus(images)
            val_loss += loss_fn(outputs, masks).item()
            val_dice += dice_coef(outputs, masks).item()
            val_iou += iou_score(outputs, masks).item()

    # Averages
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    avg_train_dice = train_dice / len(train_loader)
    avg_val_dice = val_dice / len(val_loader)
    avg_train_iou = train_iou / len(train_loader)
    avg_val_iou = val_iou / len(val_loader)

    # Scheduler step
    scheduler.step(avg_val_loss)

    # Logging
    print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"Train Dice: {avg_train_dice:.4f} | Val Dice: {avg_val_dice:.4f} | "
          f"Train IoU: {avg_train_iou:.4f} | Val IoU: {avg_val_iou:.4f}")

    # Check for improvement
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        epochs_no_improve = 0

        # ✅ Full checkpoint saving
        torch.save({
            'epoch': epoch,
            'model_state_dict': model_ImNet_Plus.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice': avg_val_dice,
            'val_loss': avg_val_loss
        }, "best_model.pth")

        print("✅ Saved new best model with optimizer and metrics")
    else:
        epochs_no_improve += 1
        print(f"⏳ No improvement for {epochs_no_improve} epochs")

    # ⛔ Early stopping
    if epochs_no_improve >= patience:
        print("⛔ Early stopping triggered")
        break

    # Update history
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_dice'].append(avg_train_dice)
    history['val_dice'].append(avg_val_dice)
    history['train_iou'].append(avg_train_iou)
    history['val_iou'].append(avg_val_iou)


### Learning Curves Plot Function

In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curves(history, metrics=None):
    if metrics is None:
        metrics = ['train_loss', 'val_loss', 'train_dice', 'val_dice', 'train_iou', 'val_iou']
    
    plt.figure(figsize=(15, 10))
    for metric in metrics:
        plt.plot(history[metric], label=metric)

    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Training and Validation Curves")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_learning_curves(history)

### Prediction Visualization

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from contextlib import nullcontext

# --- helpers ---
def eval_autocast():
    if torch.cuda.is_available():
        return torch.cuda.amp.autocast()
    elif torch.backends.mps.is_available():
        return torch.amp.autocast(device_type="mps", dtype=torch.float16)
    else:
        return nullcontext()

def denormalize_imagenet(tensor_img, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    if isinstance(tensor_img, torch.Tensor):
        tensor_img = tensor_img.detach().cpu()
    img = tensor_img.permute(1, 2, 0).numpy()
    img = img * np.array(std)[None, None, :] + np.array(mean)[None, None, :]
    img = np.clip(img, 0, 1)
    return (img * 255).astype(np.uint8)

# --- load best model ---
best_model = model_ImNet_Plus  # or create a new instance if needed
# --- load best model (handles both full checkpoint and raw state_dict) ---
ckpt = torch.load("best_model.pth", map_location=device)

# If you saved a full checkpoint: {'epoch', 'model_state_dict', 'optimizer_state_dict', ...}
state_dict = ckpt.get("model_state_dict", ckpt)  # fall back to ckpt itself if it’s already a state_dict

missing, unexpected = best_model.load_state_dict(state_dict, strict=True)
if len(missing) or len(unexpected):
    print("⚠️ load_state_dict mismatches")
    if len(missing):   print("  Missing keys:", missing)
    if len(unexpected):print("  Unexpected keys:", unexpected)

best_model.to(device).eval()

# --- visualization ---
def visualize_predictions(model, dataset, device, max_samples=10, thresh=0.5):
    n = min(len(dataset), max_samples)
    for i in tqdm(range(n), desc="Predicting"):
        image_t, mask_t = dataset[i]
        image_b = image_t.unsqueeze(0).to(device)

        with torch.no_grad():
            with eval_autocast():
                logits = model(image_b)

        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
        pred_mask = (prob > thresh).astype(np.uint8)

        img_vis = denormalize_imagenet(image_t)
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)

        overlay = img_vis.copy()
        m = pred_mask.astype(bool)
        overlay[m] = (0.5 * overlay[m] + 0.5 * np.array([0, 255, 0], dtype=np.float32)).astype(np.uint8)

        fig, axs = plt.subplots(1, 4, figsize=(16, 4))
        axs[0].imshow(img_vis);                axs[0].set_title("Image");         axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray");   axs[1].set_title("Ground Truth");  axs[1].axis("off")
        axs[2].imshow(pred_mask, cmap="gray"); axs[2].set_title("Predicted");     axs[2].axis("off")
        axs[3].imshow(overlay);                axs[3].set_title("Overlay");       axs[3].axis("off")
        plt.tight_layout(); plt.show()

# --- run ---
visualize_predictions(best_model, test_dataset, device, max_samples=10, thresh=THRESH if 'THRESH' in globals() else 0.5)


In [ ]:
# --- visualization with instance splitting ---
from skimage import segmentation

def visualize_predictions_with_instances(
    model,
    dataset,
    device,
    max_samples=10,
    thresh=0.5,
    ws_min_area=150,
    ws_open_radius=1,
    ws_gaussian_sigma=0.0,
    ws_min_peak_distance=8,
    ws_h_minima=None
):
    n = min(len(dataset), max_samples)
    for i in tqdm(range(n), desc="Predicting"):
        image_t, mask_t = dataset[i]
        image_b = image_t.unsqueeze(0).to(device)

        # forward
        with torch.no_grad():
            with eval_autocast():
                logits = model(image_b)

        # semantic mask (0/1)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
        pred_mask = (prob > thresh).astype(np.uint8)

        # --- instance split (watershed) ---
        labels = split_instances_watershed(
            pred_mask,
            min_area=ws_min_area,
            open_radius=ws_open_radius,
            gaussian_sigma=ws_gaussian_sigma,
            min_peak_distance=ws_min_peak_distance,
            h_minima=ws_h_minima
        )
        n_instances = int(labels.max())

        # pretty overlays
        img_vis = denormalize_imagenet(image_t)
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)
        inst_overlay = overlay_instances_on_image(img_vis / 255.0, labels, alpha=0.35)  # expects float image 0..1
        boundaries = segmentation.find_boundaries(labels, mode="outer")
        boundary_overlay = img_vis.copy()
        boundary_overlay[boundaries] = [255, 0, 0]  # red outlines

        # quick semantic overlay too (green)
        sem_overlay = img_vis.copy()
        m = pred_mask.astype(bool)
        sem_overlay[m] = (0.5 * sem_overlay[m] + 0.5 * np.array([0, 255, 0], dtype=np.float32)).astype(np.uint8)

        # plot
        fig, axs = plt.subplots(1, 5, figsize=(20, 4))
        axs[0].imshow(img_vis);              axs[0].set_title("Image");             axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray"); axs[1].set_title("Ground Truth");      axs[1].axis("off")
        axs[2].imshow(sem_overlay);          axs[2].set_title("Semantic Overlay");  axs[2].axis("off")
        axs[3].imshow(inst_overlay);         axs[3].set_title(f"Instances (N={n_instances})"); axs[3].axis("off")
        axs[4].imshow(boundary_overlay);     axs[4].set_title("Instance Boundaries");          axs[4].axis("off")
        plt.tight_layout(); plt.show()

        # also print count in the cell output
        print(f"[{i+1}/{n}] separated instances: {n_instances}")


### Saving The Last Epoch Trained Model

In [ ]:
import json, torch, glob, os

# ---- SAVE ----
# Try to read from dataset attributes; if missing, fall back to common filename
def _guess_ann_path(dataset):
    ann = getattr(dataset, "ann_path", None)
    if ann and os.path.isfile(ann):
        return ann
    # fallback: look for COCO json in the image dir
    cands = glob.glob(os.path.join(dataset.img_dir, "*_annotations.coco.json"))
    return cands[0] if cands else None

save_meta = {
    "img_dir": getattr(test_dataset, "img_dir", None),
    "ann_path": _guess_ann_path(test_dataset),
    "image_size": IMAGE_SIZE,
    "flip_h": getattr(test_dataset, "flip_h", False),
    "flip_v": getattr(test_dataset, "flip_v", False),
    "norm": "imagenet",              # <- set to what you used for val/test
    "model": {
        "encoder_name": "efficientnet-b3",
        "in_channels": 3,
        "classes": 1
    }
}

with open("dataset_info.json", "w") as f:
    json.dump(save_meta, f)

# Save model weights only (no optimizer needed for inference)
torch.save(model.state_dict(), "temp_model.pth")
print("✅ Saved: temp_model.pth + dataset_info.json")


### Last Epoch Model Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, jaccard_score
import numpy as np
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

# If you don't have seaborn installed, comment these two lines and use the Matplotlib heatmap below
try:
    import seaborn as sns
    _HAS_SNS = True
except Exception:
    _HAS_SNS = False

def evaluate_metrics(model, dataset, device, threshold=0.5):
    y_true_all = []
    y_pred_all = []

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(len(dataset)), desc="Evaluating"):
            image, true_mask = dataset[i]
            image_tensor = image.unsqueeze(0).to(device)

            logits = model(image_tensor)
            pred_mask = (logits.squeeze().cpu().numpy() > threshold).astype(np.uint8)

            true_mask_np = true_mask.squeeze().cpu().numpy().astype(np.uint8)

            # Flatten per-pixel labels
            y_true_all.extend(true_mask_np.ravel())
            y_pred_all.extend(pred_mask.ravel())

    # Confusion matrix (labels=[0,1] -> rows=true, cols=pred)
    cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])

    # Basic metrics
    acc       = accuracy_score(y_true_all, y_pred_all)
    precision = precision_score(y_true_all, y_pred_all, zero_division=0)
    recall    = recall_score(y_true_all, y_pred_all, zero_division=0)
    f1        = f1_score(y_true_all, y_pred_all, zero_division=0)         # == Dice for binary foreground
    iou       = jaccard_score(y_true_all, y_pred_all, zero_division=0)    # foreground class IoU

    # Dice from confusion matrix (explicit), and Dice loss
    TN, FP = cm[0, 0], cm[0, 1]
    FN, TP = cm[1, 0], cm[1, 1]
    dice = (2.0 * TP) / (2.0 * TP + FP + FN) if (2.0 * TP + FP + FN) > 0 else 0.0
    dice_loss = 1.0 - dice

    return cm, {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou,
        "dice": dice,
        "dice_loss": dice_loss
    }

# ---- Run evaluation ----
cm, metrics = evaluate_metrics(model_ImNet_Plus, test_dataset, device, threshold=0.5)

# ---- Print metrics ----
print("Confusion Matrix (rows=true, cols=pred):\n", cm)
print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall   : {metrics['recall']:.4f}")
print(f"F1 Score : {metrics['f1']:.4f}  (== Dice for binary foreground)")
print(f"IoU      : {metrics['iou']:.4f}")
print(f"Dice     : {metrics['dice']:.4f}")
print(f"Dice Loss: {metrics['dice_loss']:.4f}")

# ---- Plot confusion matrix ----
if _HAS_SNS:
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
    plt.title("Confusion Matrix - Test Set")
    plt.xlabel("Prediction"); plt.ylabel("Ground Truth")
    plt.tight_layout(); plt.show()
else:
    # Pure Matplotlib fallback
    fig, ax = plt.subplots(figsize=(5,4))
    im = ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{v}", ha="center", va="center")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Pred 0", "Pred 1"]); ax.set_yticklabels(["True 0", "True 1"])
    ax.set_xlabel("Prediction"); ax.set_ylabel("Ground Truth")
    ax.set_title("Confusion Matrix - Test Set")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()


### Best model Evaluation

In [ ]:
import torch

# Load model architecture
# Make sure you import and initialize the SAME architecture as when you trained
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b3",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None
).to(device)

# 2️⃣ Load checkpoint and extract only the model weights
checkpoint = torch.load("best_model.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])  # <- use the correct key
model.eval()
# Then run the evaluation
cm, metrics = evaluate_metrics(model, test_dataset, device, threshold=0.5)

print("Confusion Matrix (rows=true, cols=pred):\n", cm)
print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall   : {metrics['recall']:.4f}")
print(f"F1 Score : {metrics['f1']:.4f}")
print(f"IoU      : {metrics['iou']:.4f}")
print(f"Dice     : {metrics['dice']:.4f}")
print(f"Dice Loss: {metrics['dice_loss']:.4f}")


In [ ]:
# ---- Plot confusion matrix ----
if _HAS_SNS:
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
    plt.title("Confusion Matrix - Test Set")
    plt.xlabel("Prediction"); plt.ylabel("Ground Truth")
    plt.tight_layout(); plt.show()
else:
    # Pure Matplotlib fallback
    fig, ax = plt.subplots(figsize=(5,4))
    im = ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{v}", ha="center", va="center")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Pred 0", "Pred 1"]); ax.set_yticklabels(["True 0", "True 1"])
    ax.set_xlabel("Prediction"); ax.set_ylabel("Ground Truth")
    ax.set_title("Confusion Matrix - Test Set")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()


## Evaluation Results – Almond Tree Segmentation

### 1. Overview
The evaluation was performed on a test set of almond tree aerial images captured at different times of the year — including:
- **Full foliage periods** where the trees are dense with leaves
- **Blooming periods** where branches are visible with flowers
- **Bare branch stages** after leaf drop

This variability makes the segmentation task more challenging, as the model must recognize almond trees under diverse visual conditions and background patterns.

---

### 2. Confusion Matrix (rows = ground truth, cols = predictions)
|            | Predicted Non-Tree | Predicted Tree |
|------------|--------------------|----------------|
| **True Non-Tree** | 8,828,938            | 1,455,056       |
| **True Tree**     | 363,691              | 8,226,683       |

- **True Negatives (TN):** 8,828,938 pixels correctly identified as background.
- **False Positives (FP):** 1,455,056 pixels incorrectly predicted as tree (common with bare soil or shadow regions).
- **False Negatives (FN):** 363,691 pixels missed as tree (often in thin branches or low-contrast flower regions).
- **True Positives (TP):** 8,226,683 pixels correctly identified as almond tree.

---

### 3. Performance Metrics
- **Accuracy:** 90.36% – Overall pixel-level correctness.
- **Precision:** 84.97% – When the model predicts "tree," it’s correct 85% of the time.  
  _(Lower precision indicates some over-detection, possibly confusing other vegetation or shadows with almond trees.)_
- **Recall:** 95.77% – The model captures most tree pixels, showing strong sensitivity even in complex seasonal appearances.
- **F1 Score:** 90.05% – Balanced trade-off between precision and recall.
- **IoU:** 81.89% – Intersection-over-Union for the tree class, indicating solid overlap between prediction and ground truth masks.
- **Dice Score:** 90.05% – Another overlap metric, especially sensitive to class imbalance.
- **Dice Loss:** 0.0995 – Low value, showing strong segmentation performance.

---

### 4. Interpretation in Context
- The **high recall** is critical for our use case: detecting all almond tree areas to avoid missing relevant vegetation, especially in agricultural monitoring.
- Slightly lower precision is acceptable here since over-detection can be managed in downstream processing (e.g., filtering by location or seasonal growth patterns).
- Performance remains **strong despite seasonal variability**, meaning the model learned robust shape and texture cues beyond just leaf coverage.
- Most errors appear in **border regions of trees** and **scenarios with shadows or ground cover**, which can look similar to branches or leaves from above.

---

### 5. Next Steps
- **Post-processing improvements** (e.g., morphological operations, shadow removal filters) to reduce false positives.
- **Data augmentation focus** on bare-branch and flowering stages to further improve precision.
- Consider **temporal analysis** (multi-date imagery) to better distinguish permanent tree structures from seasonal changes.

---


In [ ]:
# ================== RADIAL-JUMPS ROW/COL "SAUSAGE" SPLITTER ==================
import numpy as np
from scipy import ndimage as ndi
from skimage import morphology, measure
from skimage.morphology import disk

# ---- helpers ----
def _line_kernel(angle_rad, length=31, thickness=3):
    L = int(max(3, length)); W = int(max(1, thickness))
    size = int(np.ceil(L*np.sqrt(2))) + 2*W + 3
    k = np.zeros((size, size), np.uint8); c = size//2
    s = np.linspace(-(L-1)/2, (L-1)/2, L, dtype=np.float32)
    yy = c + s*np.sin(angle_rad); xx = c + s*np.cos(angle_rad)
    rr = np.clip(np.round(yy).astype(int), 0, size-1)
    cc = np.clip(np.round(xx).astype(int), 0, size-1)
    k[rr, cc] = 1
    if W > 1:
        k = morphology.binary_dilation(k, disk(W//2)).astype(np.uint8)
    return k.astype(bool)

def _radii_from_centroid(reg_sub, cy, cx, step_deg=15, max_step=None):
    """Sample radius from centroid to boundary at angles 0..π (step_deg)."""
    Hs, Ws = reg_sub.shape
    if max_step is None:
        max_step = np.hypot(Hs, Ws)
    deg = np.arange(0, 180, step_deg, dtype=np.float32)
    ang = np.deg2rad(deg)
    radii = np.zeros_like(ang, dtype=np.float32)

    for i, a in enumerate(ang):
        dx, dy = np.cos(a), np.sin(a)
        x, y = float(cx), float(cy)
        r = 0.0
        # march until we leave the region
        while 0 <= int(round(y)) < Hs and 0 <= int(round(x)) < Ws and reg_sub[int(round(y)), int(round(x))]:
            x += dx
            y += dy
            r += 1.0
            if r > max_step:
                break
        radii[i] = r
    return deg, ang, radii

def _valleys_1d(profile, k_cuts, smooth_sigma):
    """Pick k valley indices from a 1-D profile (cheap)."""
    if k_cuts <= 0 or profile.size < 8:
        return []
    p = ndi.gaussian_filter1d(profile.astype(np.float32), smooth_sigma, mode='nearest')
    inv = -p
    is_peak = (inv > np.r_[inv[1:], -np.inf]) & (inv > np.r_[-np.inf, inv[:-1]])
    idx = np.where(is_peak)[0]
    if idx.size == 0:
        return []
    # pick the deepest valleys
    order = np.argsort(p[idx])
    idx = idx[order[:k_cuts]]
    idx.sort()
    return idx.tolist()

# ---- main splitter (keeps your old signature so your viz code works) ----
def split_instances_geom_shape_rowaware(
    pred_mask,
    image_rgb=None,
    # cleanup
    min_area=220, open_radius=1,
    # (compat; unused)
    neck_rel=0.28, neck_dilate=4, neck_down=3,
    seed_radius=3, min_peak_distance=4, h_minima=1.1,
    gaussian_sigma=0.5, dist_gamma=1.25,
    compactness=5.0, edge_weight=0.36,
    row_cut_len=0, row_cut_thick=1,
    col_cut_len=0, col_cut_thick=1,
    # NEW core knobs for this “radial-jumps” method
    step_deg=15,
    jump_factor=1.45,
    radius_q=(0.35, 0.65),
    spacing_scale=0.98,
    band_frac=0.10,
    valley_smooth=0.02,
    bins_min=48, bins_max=256,
    # --------- COMPATIBILITY ALIASES (so your old visualize() call works) ---------
    ar_trigger=None,             # ignored; kept to avoid TypeError
    band_frac_row=None,          # if given, mapped to band_frac
    band_frac_col=None           # if given, mapped to band_frac
):
    # ---- map legacy names to the new ones (if provided) ----
    if (band_frac_row is not None) or (band_frac_col is not None):
        vals = [v for v in [band_frac_row, band_frac_col] if v is not None]
        if vals:
            band_frac = float(np.mean(vals))
    # ---- pre-clean ----
    m = pred_mask.astype(bool)
    if open_radius and open_radius > 0:
        m = morphology.opening(m, disk(int(open_radius)))
    m = morphology.remove_small_objects(m, min_size=int(min_area))
    if not m.any():
        return np.zeros_like(pred_mask, np.int32)

    labeled = measure.label(m, connectivity=1)

    # ----- first pass: per-object base radius & jump directions -----
    base_radii = []
    jump_angles = []   # degrees in [0,180)
    obj_info = []      # cache bbox + centroid for second pass

    for r in measure.regionprops(labeled):
        if r.area < min_area:
            continue
        minr, minc, maxr, maxc = r.bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == r.label)

        cy, cx = r.centroid
        cy -= minr; cx -= minc

        deg, ang, radii = _radii_from_centroid(reg_sub, cy, cx, step_deg=step_deg)

        # robust per-object base radius from middle quantiles
        lo, hi = np.quantile(radii, radius_q)
        base = 0.5*(lo + hi)
        base_radii.append(base)

        # jumps: angles where the radius is much larger than base
        jump_idx = np.where(radii > (jump_factor * base))[0]
        if jump_idx.size:
            # compress opposite directions: map to [0,180)
            jump_angles.extend(deg[jump_idx].tolist())

        obj_info.append((r.label, (minr, minc, maxr, maxc), (cy, cx), deg, ang, radii))

    # global radius estimate = most common range → use robust median
    R_est = float(np.median(base_radii)) if base_radii else 3.0
    R_est = max(1.0, R_est)

    # global row/col angles from jump histogram (fallback to tensor if empty)
    if len(jump_angles) >= 3:
        # histogram on [0,180)
        hist, edges = np.histogram(jump_angles, bins=36, range=(0,180))
        peak_deg = float(0.5*(edges[np.argmax(hist)] + edges[np.argmax(hist)+1]))
        ang_row = np.deg2rad(peak_deg)
    else:
        # cheap fallback: structure tensor on background
        bg = (~m).astype(np.float32)
        g = ndi.gaussian_filter(bg, 2.0)
        gy, gx = np.gradient(g)
        Jxx = ndi.gaussian_filter(gx*gx, 2.0)
        Jxy = ndi.gaussian_filter(gx*gy, 2.0)
        Jyy = ndi.gaussian_filter(gy*gy, 2.0)
        theta = 0.5*np.arctan2(2*Jxy, (Jxx - Jyy + 1e-8))
        vals = theta[bg > np.percentile(bg, 50)]
        ang_row = float(np.median(vals)) if vals.size else 0.0

    ang_col = (ang_row + np.pi/2.0) % np.pi
    cos_r, sin_r = np.cos(ang_row), np.sin(ang_row)
    cos_c, sin_c = np.cos(ang_col), np.sin(ang_col)

    # ----- second pass: cut elongated blobs along row/col using valley positions -----
    cut_global = np.zeros_like(m, bool)

    for (lbl, bbox, (cy, cx), deg, ang, radii) in obj_info:
        minr, minc, maxr, maxc = bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == lbl)
        Hs, Ws = reg_sub.shape

        # coords grid in subimage
        yy, xx = np.mgrid[0:Hs, 0:Ws]
        yy_abs, xx_abs = yy + minr, xx + minc

        # projections
        t_row = xx_abs * cos_r + yy_abs * sin_r     # along rows
        s_col = -xx_abs * sin_r + yy_abs * cos_r    # across rows

        # spans inside this object
        t_vals = t_row[reg_sub]; s_vals = s_col[reg_sub]
        t_min, t_max = float(t_vals.min()), float(t_vals.max())
        s_min, s_max = float(s_vals.min()), float(s_vals.max())
        Lr = t_max - t_min; Wc = s_max - s_min + 1e-6

        # same for column axis
        t_col = xx_abs * cos_c + yy_abs * sin_c     # along columns
        s_row = -xx_abs * sin_c + yy_abs * cos_c    # across columns
        tc_vals = t_col[reg_sub]; sr_vals = s_row[reg_sub]
        tc_min, tc_max = float(tc_vals.min()), float(tc_vals.max())
        sr_min, sr_max = float(sr_vals.min()), float(sr_vals.max())
        Lc = tc_max - tc_min; Wr = sr_max - sr_min + 1e-6

        # expected single-tree diameter
        D = 2.0 * R_est * float(spacing_scale)
        # estimated number of trees along each axis
        n_row = int(np.round(Lr / max(D, 1.0)))
        n_col = int(np.round(Lc / max(D, 1.0)))

        cut_local = np.zeros_like(reg_sub, bool)

        # A) cut along row direction if elongated
        if n_row >= 2 and (Lr / Wc) >= 1.2:
            nb = int(np.clip(np.round(Lr), bins_min, bins_max))
            idx = np.clip(((t_row - t_min) / (Lr + 1e-6) * nb).astype(int), 0, nb-1)
            prof = np.bincount(idx[reg_sub], minlength=nb)   # width profile across rows
            k = max(1, n_row - 1)
            valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
            if valleys:
                # convert valley bins to t thresholds
                t_bounds = [t_min + (vi + 0.5) * (Lr / nb) for vi in valleys]
                half = max(1.0, float(band_frac) * Wc)
                for tb in t_bounds:
                    cut_local |= (np.abs(t_row - tb) <= half)

        # B) cut along column direction if elongated
        if n_col >= 2 and (Lc / Wr) >= 1.2:
            nb = int(np.clip(np.round(Lc), bins_min, bins_max))
            idx = np.clip(((t_col - tc_min) / (Lc + 1e-6) * nb).astype(int), 0, nb-1)
            prof = np.bincount(idx[reg_sub], minlength=nb)   # length profile across columns
            k = max(1, n_col - 1)
            valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
            if valleys:
                t_bounds = [tc_min + (vi + 0.5) * (Lc / nb) for vi in valleys]
                half = max(1.0, float(band_frac) * Wr)
                for tb in t_bounds:
                    cut_local |= (np.abs(t_col - tb) <= half)

        cut_global[minr:maxr, minc:maxc] |= (cut_local & reg_sub)

    # apply cuts + relabel
    if cut_global.any():
        m = m & (~cut_global)

    m = morphology.remove_small_objects(m, min_size=int(min_area))
    labels = measure.label(m, connectivity=1).astype(np.int32)
    return labels



# ---------- helper for visualization ----------
def overlay_instances_on_image(image_rgb_uint8, labels, alpha=0.35):
    from skimage import color
    img_f = np.clip(image_rgb_uint8.astype(np.float32) / 255.0, 0, 1)
    vis = color.label2rgb(labels, image=img_f, bg_label=0, alpha=float(alpha))
    return (np.clip(vis, 0, 1) * 255).astype(np.uint8)

# ---------- visualization (fits the geometric “sausage” splitter) ----------
def visualize_predictions_with_instances_geom(
    model,
    dataset,
    device,
    max_samples=10,
    thresh=0.65,
    # splitter knobs (kept for API compatibility; unused by pure-geom)
    min_area=220, open_radius=1,
    neck_rel=0.25, neck_dilate=4, neck_down=3,
    seed_radius=3, min_peak_distance=4,
    gaussian_sigma=0.5, dist_gamma=1.25,
    compactness=5.0, edge_weight=0.36,
    row_cut_len=0, row_cut_thick=1,
    col_cut_len=0, col_cut_thick=1,
    h_minima=1.0,
    # NEW: geometric sausage parameters
    ar_trigger=1.7,
    band_frac_row=0.09,
    band_frac_col=0.09,
    bins_min=48,
    bins_max=256,
):
    import torch, matplotlib.pyplot as plt
    from tqdm import tqdm
    from skimage import segmentation as _seg

    model.eval()

    n = min(len(dataset), max_samples)
    for i in tqdm(range(n), desc="Predicting + geometric sausage split"):
        image_t, mask_t = dataset[i]
        image_b = image_t.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(image_b)

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred_mask = (prob > float(thresh)).astype(np.uint8)
        img_vis = denormalize_imagenet(image_t)  # your existing denorm

        labels = split_instances_geom_shape_rowaware(
            pred_mask,
            min_area=220,
            open_radius=1,
            step_deg=15,
            jump_factor=1.45,
            spacing_scale=0.98,
            band_frac=0.04,          # ↓ from 0.10 → thinner stripes
            valley_smooth=0.02,
            bins_min=48, bins_max=256
        )


        n_instances = int(labels.max())
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)

        sem_overlay = img_vis.copy()
        m = pred_mask.astype(bool)
        sem_overlay[m] = (0.5 * sem_overlay[m] + 0.5 * np.array([0, 255, 0], dtype=np.float32)).astype(np.uint8)

        inst_overlay = overlay_instances_on_image(img_vis, labels, alpha=0.35)
        boundaries = _seg.find_boundaries(labels, mode="outer")
        boundary_overlay = img_vis.copy()
        boundary_overlay[boundaries] = [255, 0, 0]

        fig, axs = plt.subplots(1, 5, figsize=(20, 4))
        axs[0].imshow(img_vis);              axs[0].set_title("Image");                 axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray"); axs[1].set_title("Ground Truth");          axs[1].axis("off")
        axs[2].imshow(sem_overlay);          axs[2].set_title("Semantic Overlay");      axs[2].axis("off")
        axs[3].imshow(inst_overlay);         axs[3].set_title(f"Instances (N={n_instances})"); axs[3].axis("off")
        axs[4].imshow(boundary_overlay);     axs[4].set_title("Instance Boundaries");   axs[4].axis("off")
        plt.tight_layout(); plt.show()

        print(f"[{i+1}/{n}] separated instances: {n_instances}")

# =================== example call (now passes the new geom knobs) ===================
visualize_predictions_with_instances_geom(
    best_model, test_dataset, device,
    max_samples=10, thresh=0.66,
    min_area=220, open_radius=1,
    ar_trigger=1.0,          # lower = more aggressive splitting
    band_frac_row=0.09,
    band_frac_col=0.09,
    bins_min=48, bins_max=256
)


In [ ]:
# ================== RADIAL-JUMPS ROW/COL "SAUSAGE" SPLITTER (with thin cuts) ==================
import numpy as np
from scipy import ndimage as ndi
from skimage import morphology, measure
from skimage.morphology import disk, thin

# ---- helpers ----
def _line_kernel(angle_rad, length=31, thickness=3):
    L = int(max(3, length)); W = int(max(1, thickness))
    size = int(np.ceil(L*np.sqrt(2))) + 2*W + 3
    k = np.zeros((size, size), np.uint8); c = size//2
    s = np.linspace(-(L-1)/2, (L-1)/2, L, dtype=np.float32)
    yy = c + s*np.sin(angle_rad); xx = c + s*np.cos(angle_rad)
    rr = np.clip(np.round(yy).astype(int), 0, size-1)
    cc = np.clip(np.round(xx).astype(int), 0, size-1)
    k[rr, cc] = 1
    if W > 1:
        k = morphology.binary_dilation(k, disk(W//2)).astype(np.uint8)
    return k.astype(bool)

def _radii_from_centroid(reg_sub, cy, cx, step_deg=15, max_step=None):
    """Sample radius from centroid to boundary at angles 0..π (step_deg)."""
    Hs, Ws = reg_sub.shape
    if max_step is None:
        max_step = np.hypot(Hs, Ws)
    deg = np.arange(0, 180, step_deg, dtype=np.float32)
    ang = np.deg2rad(deg)
    radii = np.zeros_like(ang, dtype=np.float32)

    for i, a in enumerate(ang):
        dx, dy = np.cos(a), np.sin(a)
        x, y = float(cx), float(cy)
        r = 0.0
        while 0 <= int(round(y)) < Hs and 0 <= int(round(x)) < Ws and reg_sub[int(round(y)), int(round(x))]:
            x += dx; y += dy; r += 1.0
            if r > max_step:
                break
        radii[i] = r
    return deg, ang, radii

def _valleys_1d(profile, k_cuts, smooth_sigma):
    """Pick k valley indices from a 1-D profile (cheap)."""
    if k_cuts <= 0 or profile.size < 8:
        return []
    p = ndi.gaussian_filter1d(profile.astype(np.float32), smooth_sigma, mode='nearest')
    inv = -p
    is_peak = (inv > np.r_[inv[1:], -np.inf]) & (inv > np.r_[-np.inf, inv[:-1]])
    idx = np.where(is_peak)[0]
    if idx.size == 0:
        return []
    order = np.argsort(p[idx])
    idx = idx[order[:k_cuts]]
    idx.sort()
    return idx.tolist()

# ---- main splitter (keeps your old signature so your viz code works) ----
def split_instances_geom_shape_rowaware(
    pred_mask,
    image_rgb=None,
    # cleanup
    min_area=220, open_radius=1,
    # (compat; unused)
    neck_rel=0.28, neck_dilate=4, neck_down=3,
    seed_radius=3, min_peak_distance=4, h_minima=1.1,
    gaussian_sigma=0.5, dist_gamma=1.25,
    compactness=5.0, edge_weight=0.36,
    row_cut_len=0, row_cut_thick=1,
    col_cut_len=0, col_cut_thick=1,
    # RADIAL-JUMPS core knobs
    step_deg=15,
    jump_factor=1.45,
    radius_q=(0.35, 0.65),
    spacing_scale=0.98,
    band_frac=0.10,          # used if cut_px is None
    valley_smooth=0.02,
    bins_min=48, bins_max=256,
    # NEW: thin cuts controller
    cut_px=None,             # exact cut thickness (px). If set, overrides band_frac
    thin_cuts=True,          # skeletonize cut bands to 1 px, then redilate to cut_px
    min_cut_px=1.0,          # never thinner than 1 px total
    # compatibility aliases
    ar_trigger=None, band_frac_row=None, band_frac_col=None
):
    # map legacy row/col frac to a single frac (if user passes those)
    if (band_frac_row is not None) or (band_frac_col is not None):
        vals = [v for v in (band_frac_row, band_frac_col) if v is not None]
        if vals:
            band_frac = float(np.mean(vals))

    # helper → compute half-thickness
    def _half(span, cut_px, band_frac, min_cut_px):
        if cut_px is not None:
            return max(min_cut_px/2.0, float(cut_px)/2.0)
        return max(min_cut_px/2.0, float(band_frac) * float(span))

    # ---- pre-clean ----
    m = pred_mask.astype(bool)
    if open_radius and open_radius > 0:
        m = morphology.opening(m, disk(int(open_radius)))
    m = morphology.remove_small_objects(m, min_size=int(min_area))
    if not m.any():
        return np.zeros_like(pred_mask, np.int32)

    labeled = measure.label(m, connectivity=1)

    # ----- first pass: per-object base radius & jump directions -----
    base_radii, jump_angles, obj_info = [], [], []

    for r in measure.regionprops(labeled):
        if r.area < min_area:
            continue
        minr, minc, maxr, maxc = r.bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == r.label)

        cy, cx = r.centroid
        cy -= minr; cx -= minc

        deg, ang, radii = _radii_from_centroid(reg_sub, cy, cx, step_deg=step_deg)

        lo, hi = np.quantile(radii, radius_q)
        base = 0.5*(lo + hi)
        base_radii.append(base)

        jump_idx = np.where(radii > (jump_factor * base))[0]
        if jump_idx.size:
            jump_angles.extend(deg[jump_idx].tolist())

        obj_info.append((r.label, (minr, minc, maxr, maxc), (cy, cx), deg, ang, radii))

    # global radius estimate
    R_est = float(np.median(base_radii)) if base_radii else 3.0
    R_est = max(1.0, R_est)

    # global row/col angles
    if len(jump_angles) >= 3:
        hist, edges = np.histogram(jump_angles, bins=36, range=(0,180))
        peak_deg = float(0.5*(edges[np.argmax(hist)] + edges[np.argmax(hist)+1]))
        ang_row = np.deg2rad(peak_deg)
    else:
        bg = (~m).astype(np.float32)
        g = ndi.gaussian_filter(bg, 2.0)
        gy, gx = np.gradient(g)
        Jxx = ndi.gaussian_filter(gx*gx, 2.0)
        Jxy = ndi.gaussian_filter(gx*gy, 2.0)
        Jyy = ndi.gaussian_filter(gy*gy, 2.0)
        theta = 0.5*np.arctan2(2*Jxy, (Jxx - Jyy + 1e-8))
        vals = theta[bg > np.percentile(bg, 50)]
        ang_row = float(np.median(vals)) if vals.size else 0.0

    ang_col = (ang_row + np.pi/2.0) % np.pi
    cos_r, sin_r = np.cos(ang_row), np.sin(ang_row)
    cos_c, sin_c = np.cos(ang_col), np.sin(ang_col)

    # ----- second pass: cut elongated blobs along row/col using valley positions -----
    cut_global = np.zeros_like(m, bool)

    for (lbl, bbox, (cy, cx), deg, ang, radii) in obj_info:
        minr, minc, maxr, maxc = bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == lbl)
        Hs, Ws = reg_sub.shape

        yy, xx = np.mgrid[0:Hs, 0:Ws]
        yy_abs, xx_abs = yy + minr, xx + minc

        # row axis
        t_row = xx_abs * cos_r + yy_abs * sin_r
        s_col = -xx_abs * sin_r + yy_abs * cos_r
        t_vals = t_row[reg_sub]; s_vals = s_col[reg_sub]
        t_min, t_max = float(t_vals.min()), float(t_vals.max())
        s_min, s_max = float(s_vals.min()), float(s_vals.max())
        Lr = t_max - t_min; Wc = s_max - s_min + 1e-6

        # column axis
        t_col = xx_abs * cos_c + yy_abs * sin_c
        s_row = -xx_abs * sin_c + yy_abs * cos_c
        tc_vals = t_col[reg_sub]; sr_vals = s_row[reg_sub]
        tc_min, tc_max = float(tc_vals.min()), float(tc_vals.max())
        sr_min, sr_max = float(sr_vals.min()), float(sr_vals.max())
        Lc = tc_max - tc_min; Wr = sr_max - sr_min + 1e-6

        D = 2.0 * R_est * float(spacing_scale)
        n_row = int(np.round(Lr / max(D, 1.0)))
        n_col = int(np.round(Lc / max(D, 1.0)))

        cut_local = np.zeros_like(reg_sub, bool)

        # A) along rows
        if n_row >= 2 and (Lr / Wc) >= 1.2:
            nb = int(np.clip(np.round(Lr), bins_min, bins_max))
            idx = np.clip(((t_row - t_min) / (Lr + 1e-6) * nb).astype(int), 0, nb-1)
            prof = np.bincount(idx[reg_sub], minlength=nb)
            k = max(1, n_row - 1)
            valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
            if valleys:
                t_bounds = [t_min + (vi + 0.5) * (Lr / nb) for vi in valleys]
                half = _half(Wc, cut_px, band_frac, min_cut_px)
                for tb in t_bounds:
                    cut_local |= (np.abs(t_row - tb) <= half)

        # B) along columns
        if n_col >= 2 and (Lc / Wr) >= 1.2:
            nb = int(np.clip(np.round(Lc), bins_min, bins_max))
            idx = np.clip(((t_col - tc_min) / (Lc + 1e-6) * nb).astype(int), 0, nb-1)
            prof = np.bincount(idx[reg_sub], minlength=nb)
            k = max(1, n_col - 1)
            valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
            if valleys:
                t_bounds = [tc_min + (vi + 0.5) * (Lc / nb) for vi in valleys]
                half = _half(Wr, cut_px, band_frac, min_cut_px)
                for tb in t_bounds:
                    cut_local |= (np.abs(t_col - tb) <= half)

        # --- make bands thin if requested ---
        if thin_cuts and cut_local.any():
            cut_local = thin(cut_local)  # 1 px skeleton
            if (cut_px is not None) and (cut_px > 1.0):
                rad = int(max(0, np.round(cut_px/2.0) - 1))
                if rad > 0:
                    cut_local = morphology.binary_dilation(cut_local, disk(rad))

        cut_global[minr:maxr, minc:maxc] |= (cut_local & reg_sub)

    # apply cuts + relabel
    if cut_global.any():
        m = m & (~cut_global)

    m = morphology.remove_small_objects(m, min_size=int(min_area))
    labels = measure.label(m, connectivity=1).astype(np.int32)
    return labels


# ---------- helper for visualization ----------
def overlay_instances_on_image(image_rgb_uint8, labels, alpha=0.35):
    from skimage import color
    img_f = np.clip(image_rgb_uint8.astype(np.float32) / 255.0, 0, 1)
    vis = color.label2rgb(labels, image=img_f, bg_label=0, alpha=float(alpha))
    return (np.clip(vis, 0, 1) * 255).astype(np.uint8)

# ---------- visualization (unchanged; pass new thin-cut knobs if you want) ----------
def visualize_predictions_with_instances_geom(
    model, dataset, device,
    max_samples=10, thresh=0.65,
    # legacy compat knobs (unused by this splitter)
    min_area=220, open_radius=1,
    neck_rel=0.25, neck_dilate=4, neck_down=3,
    seed_radius=3, min_peak_distance=4,
    gaussian_sigma=0.5, dist_gamma=1.25,
    compactness=5.0, edge_weight=0.36,
    row_cut_len=0, row_cut_thick=1,
    col_cut_len=0, col_cut_thick=1,
    h_minima=1.0,
    # sausage params (if you still pass row/col fracs, they'll be mapped)
    ar_trigger=1.7,
    band_frac_row=0.09, band_frac_col=0.09,
    bins_min=48, bins_max=256
):
    import torch, matplotlib.pyplot as plt
    from tqdm import tqdm
    from skimage import segmentation as _seg

    model.eval()
    n = min(len(dataset), max_samples)

    for i in tqdm(range(n), desc="Predicting + geometric sausage split"):
        image_t, mask_t = dataset[i]
        image_b = image_t.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(image_b)

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred_mask = (prob > float(thresh)).astype(np.uint8)
        img_vis = denormalize_imagenet(image_t)

        labels = split_instances_geom_shape_rowaware(
            pred_mask,
            min_area=int(min_area), open_radius=int(open_radius),
            # radial-jumps / thin-cut knobs
            step_deg=15, jump_factor=1.45,
            spacing_scale=0.98,
            # choose one of the two:
            # 1) pixel-exact thin cut
            cut_px=1.5, thin_cuts=True,   # ~1–2 px visual thickness
            # 2) or fractional (comment the two lines above and use this):
            # band_frac=0.04,
            valley_smooth=0.02,
            bins_min=int(bins_min), bins_max=int(bins_max),
            # legacy mapping (if you still pass these)
            band_frac_row=float(band_frac_row), band_frac_col=float(band_frac_col),
        )

        n_instances = int(labels.max())
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)

        sem_overlay = img_vis.copy()
        m = pred_mask.astype(bool)
        sem_overlay[m] = (0.5 * sem_overlay[m] + 0.5 * np.array([0, 255, 0], dtype=np.float32)).astype(np.uint8)

        inst_overlay = overlay_instances_on_image(img_vis, labels, alpha=0.35)
        boundaries = _seg.find_boundaries(labels, mode="outer")
        boundary_overlay = img_vis.copy()
        boundary_overlay[boundaries] = [255, 0, 0]

        fig, axs = plt.subplots(1, 5, figsize=(20, 4))
        axs[0].imshow(img_vis);              axs[0].set_title("Image");                 axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray"); axs[1].set_title("Ground Truth");          axs[1].axis("off")
        axs[2].imshow(sem_overlay);          axs[2].set_title("Semantic Overlay");      axs[2].axis("off")
        axs[3].imshow(inst_overlay);         axs[3].set_title(f"Instances (N={n_instances})"); axs[3].axis("off")
        axs[4].imshow(boundary_overlay);     axs[4].set_title("Instance Boundaries");   axs[4].axis("off")
        plt.tight_layout(); plt.show()

        print(f"[{i+1}/{n}] separated instances: {n_instances}")


# =================== example call ===================
visualize_predictions_with_instances_geom(
    best_model, test_dataset, device,
    max_samples=10, thresh=0.66,
    min_area=220, open_radius=1,
    # if you want fractional control instead, comment cut_px in the viz and use these:
    band_frac_row=0.04, band_frac_col=0.04,
    bins_min=48, bins_max=256
)


In [ ]:
# ================== RADIAL-JUMPS ROW/COL "SAUSAGE" SPLITTER (with thin cuts + 2nd pass) ==================
import numpy as np
from scipy import ndimage as ndi
from skimage import morphology, measure
from skimage.morphology import disk, thin

# ---- helpers ----
def _line_kernel(angle_rad, length=31, thickness=3):
    L = int(max(3, length)); W = int(max(1, thickness))
    size = int(np.ceil(L*np.sqrt(2))) + 2*W + 3
    k = np.zeros((size, size), np.uint8); c = size//2
    s = np.linspace(-(L-1)/2, (L-1)/2, L, dtype=np.float32)
    yy = c + s*np.sin(angle_rad); xx = c + s*np.cos(angle_rad)
    rr = np.clip(np.round(yy).astype(int), 0, size-1)
    cc = np.clip(np.round(xx).astype(int), 0, size-1)
    k[rr, cc] = 1
    if W > 1:
        k = morphology.binary_dilation(k, disk(W//2)).astype(np.uint8)
    return k.astype(bool)

def _radii_from_centroid(reg_sub, cy, cx, step_deg=15, max_step=None):
    """Sample radius from centroid to boundary at angles 0..π (step_deg)."""
    Hs, Ws = reg_sub.shape
    if max_step is None:
        max_step = np.hypot(Hs, Ws)
    deg = np.arange(0, 180, step_deg, dtype=np.float32)
    ang = np.deg2rad(deg)
    radii = np.zeros_like(ang, dtype=np.float32)

    for i, a in enumerate(ang):
        dx, dy = np.cos(a), np.sin(a)
        x, y = float(cx), float(cy)
        r = 0.0
        while 0 <= int(round(y)) < Hs and 0 <= int(round(x)) < Ws and reg_sub[int(round(y)), int(round(x))]:
            x += dx; y += dy; r += 1.0
            if r > max_step:
                break
        radii[i] = r
    return deg, ang, radii

def _valleys_1d(profile, k_cuts, smooth_sigma):
    """Pick k valley indices from a 1-D profile (cheap)."""
    if k_cuts <= 0 or profile.size < 8:
        return []
    p = ndi.gaussian_filter1d(profile.astype(np.float32), smooth_sigma, mode='nearest')
    inv = -p
    is_peak = (inv > np.r_[inv[1:], -np.inf]) & (inv > np.r_[-np.inf, inv[:-1]])
    idx = np.where(is_peak)[0]
    if idx.size == 0:
        return []
    order = np.argsort(p[idx])
    idx = idx[order[:k_cuts]]
    idx.sort()
    return idx.tolist()

# ---- main splitter (keeps your old signature so your viz code works) ----
def split_instances_geom_shape_rowaware(
    pred_mask,
    image_rgb=None,
    # cleanup
    min_area=220, open_radius=1,
    # (compat; unused)
    neck_rel=0.28, neck_dilate=4, neck_down=3,
    seed_radius=3, min_peak_distance=4, h_minima=1.1,
    gaussian_sigma=0.5, dist_gamma=1.25,
    compactness=5.0, edge_weight=0.36,
    row_cut_len=0, row_cut_thick=1,
    col_cut_len=0, col_cut_thick=1,
    # RADIAL-JUMPS core knobs
    step_deg=15,
    jump_factor=1.45,
    radius_q=(0.35, 0.65),
    spacing_scale=0.98,
    band_frac=0.10,          # used if cut_px is None
    valley_smooth=0.02,
    bins_min=48, bins_max=256,
    # NEW: thin cuts controller
    cut_px=None,             # exact cut thickness (px). If set, overrides band_frac
    thin_cuts=True,          # skeletonize cut bands to 1 px, then redilate to cut_px
    min_cut_px=1.0,          # never thinner than 1 px total
    # compatibility aliases
    ar_trigger=None, band_frac_row=None, band_frac_col=None
):
    # map legacy row/col frac to a single frac (if user passes those)
    if (band_frac_row is not None) or (band_frac_col is not None):
        vals = [v for v in (band_frac_row, band_frac_col) if v is not None]
        if vals:
            band_frac = float(np.mean(vals))

    # helper → compute half-thickness
    def _half(span, cut_px, band_frac, min_cut_px):
        if cut_px is not None:
            return max(min_cut_px/2.0, float(cut_px)/2.0)
        return max(min_cut_px/2.0, float(band_frac) * float(span))

    # ---- pre-clean ----
    m = pred_mask.astype(bool)
    if open_radius and open_radius > 0:
        m = morphology.opening(m, disk(int(open_radius)))
    m = morphology.remove_small_objects(m, min_size=int(min_area))
    if not m.any():
        return np.zeros_like(pred_mask, np.int32)

    labeled = measure.label(m, connectivity=1)

    # ----- first pass: per-object base radius & jump directions -----
    base_radii, jump_angles, obj_info = [], [], []

    for r in measure.regionprops(labeled):
        if r.area < min_area:
            continue
        minr, minc, maxr, maxc = r.bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == r.label)

        cy, cx = r.centroid
        cy -= minr; cx -= minc

        deg, ang, radii = _radii_from_centroid(reg_sub, cy, cx, step_deg=step_deg)

        lo, hi = np.quantile(radii, radius_q)
        base = 0.5*(lo + hi)
        base_radii.append(base)

        jump_idx = np.where(radii > (jump_factor * base))[0]
        if jump_idx.size:
            jump_angles.extend(deg[jump_idx].tolist())

        obj_info.append((r.label, (minr, minc, maxr, maxc), (cy, cx), deg, ang, radii))

    # global radius estimate
    R_est = float(np.median(base_radii)) if base_radii else 3.0
    R_est = max(1.0, R_est)

    # global row/col angles
    if len(jump_angles) >= 3:
        hist, edges = np.histogram(jump_angles, bins=36, range=(0,180))
        peak_deg = float(0.5*(edges[np.argmax(hist)] + edges[np.argmax(hist)+1]))
        ang_row = np.deg2rad(peak_deg)
    else:
        bg = (~m).astype(np.float32)
        g = ndi.gaussian_filter(bg, 2.0)
        gy, gx = np.gradient(g)
        Jxx = ndi.gaussian_filter(gx*gx, 2.0)
        Jxy = ndi.gaussian_filter(gx*gy, 2.0)
        Jyy = ndi.gaussian_filter(gy*gy, 2.0)
        theta = 0.5*np.arctan2(2*Jxy, (Jxx - Jyy + 1e-8))
        vals = theta[bg > np.percentile(bg, 50)]
        ang_row = float(np.median(vals)) if vals.size else 0.0

    ang_col = (ang_row + np.pi/2.0) % np.pi
    cos_r, sin_r = np.cos(ang_row), np.sin(ang_row)
    cos_c, sin_c = np.cos(ang_col), np.sin(ang_col)

    # ----- second stage: cut elongated blobs along row/col using valley positions -----
    cut_global = np.zeros_like(m, bool)

    for (lbl, bbox, (cy, cx), deg, ang, radii) in obj_info:
        minr, minc, maxr, maxc = bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == lbl)
        Hs, Ws = reg_sub.shape

        yy, xx = np.mgrid[0:Hs, 0:Ws]
        yy_abs, xx_abs = yy + minr, xx + minc

        # row axis
        t_row = xx_abs * cos_r + yy_abs * sin_r
        s_col = -xx_abs * sin_r + yy_abs * cos_r
        t_vals = t_row[reg_sub]; s_vals = s_col[reg_sub]
        t_min, t_max = float(t_vals.min()), float(t_vals.max())
        s_min, s_max = float(s_vals.min()), float(s_vals.max())
        Lr = t_max - t_min; Wc = s_max - s_min + 1e-6

        # column axis
        t_col = xx_abs * cos_c + yy_abs * sin_c
        s_row = -xx_abs * sin_c + yy_abs * cos_c
        tc_vals = t_col[reg_sub]; sr_vals = s_row[reg_sub]
        tc_min, tc_max = float(tc_vals.min()), float(tc_vals.max())
        sr_min, sr_max = float(sr_vals.min()), float(sr_vals.max())
        Lc = tc_max - tc_min; Wr = sr_max - sr_min + 1e-6

        D = 2.0 * R_est * float(spacing_scale)
        n_row = int(np.round(Lr / max(D, 1.0)))
        n_col = int(np.round(Lc / max(D, 1.0)))

        cut_local = np.zeros_like(reg_sub, bool)

        # A) along rows
        if n_row >= 2 and (Lr / Wc) >= 1.2:
            nb = int(np.clip(np.round(Lr), bins_min, bins_max))
            idx = np.clip(((t_row - t_min) / (Lr + 1e-6) * nb).astype(int), 0, nb-1)
            prof = np.bincount(idx[reg_sub], minlength=nb)
            k = max(1, n_row - 1)
            valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
            if valleys:
                t_bounds = [t_min + (vi + 0.5) * (Lr / nb) for vi in valleys]
                half = _half(Wc, cut_px, band_frac, min_cut_px)
                for tb in t_bounds:
                    cut_local |= (np.abs(t_row - tb) <= half)

        # B) along columns
        if n_col >= 2 and (Lc / Wr) >= 1.2:
            nb = int(np.clip(np.round(Lc), bins_min, bins_max))
            idx = np.clip(((t_col - tc_min) / (Lc + 1e-6) * nb).astype(int), 0, nb-1)
            prof = np.bincount(idx[reg_sub], minlength=nb)
            k = max(1, n_col - 1)
            valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
            if valleys:
                t_bounds = [tc_min + (vi + 0.5) * (Lc / nb) for vi in valleys]
                half = _half(Wr, cut_px, band_frac, min_cut_px)
                for tb in t_bounds:
                    cut_local |= (np.abs(t_col - tb) <= half)

        # --- make bands thin if requested ---
        if thin_cuts and cut_local.any():
            cut_local = thin(cut_local)  # 1 px skeleton
            if (cut_px is not None) and (cut_px > 1.0):
                rad = int(max(0, np.round(cut_px/2.0) - 1))
                if rad > 0:
                    cut_local = morphology.binary_dilation(cut_local, disk(rad))

        cut_global[minr:maxr, minc:maxc] |= (cut_local & reg_sub)

    # apply cuts + relabel
    if cut_global.any():
        m = m & (~cut_global)

    m = morphology.remove_small_objects(m, min_size=int(min_area))
    labels = measure.label(m, connectivity=1).astype(np.int32)
    return labels


# ---------- helper for visualization ----------
def overlay_instances_on_image(image_rgb_uint8, labels, alpha=0.35):
    from skimage import color
    img_f = np.clip(image_rgb_uint8.astype(np.float32) / 255.0, 0, 1)
    vis = color.label2rgb(labels, image=img_f, bg_label=0, alpha=float(alpha))
    return (np.clip(vis, 0, 1) * 255).astype(np.uint8)

# ---------- visualization WITH A SECOND PASS (aggressive finish) ----------
def visualize_predictions_with_instances_geom(
    model, dataset, device,
    max_samples=15, thresh=0.65,
    # legacy compat knobs (unused by this splitter)
    min_area=220, open_radius=1,
    neck_rel=0.25, neck_dilate=4, neck_down=3,
    seed_radius=3, min_peak_distance=4,
    gaussian_sigma=0.5, dist_gamma=1.25,
    compactness=5.0, edge_weight=0.36,
    row_cut_len=0, row_cut_thick=1,
    col_cut_len=0, col_cut_thick=1,
    h_minima=1.0,
    # first-pass sausage params
    bins_min=48, bins_max=256,
    # second-pass toggle & knobs
    second_pass=True,
    sp_step_deg=10,
    sp_jump_factor=1.28,
    sp_radius_q=(0.25, 0.55),
    sp_spacing_scale=0.90,
    sp_valley_smooth=0.012,
    sp_bins_min=64, sp_bins_max=384,
    sp_cut_px=2.0, sp_thin_cuts=True
):
    import torch, matplotlib.pyplot as plt
    from tqdm import tqdm
    from skimage import segmentation as _seg

    model.eval()
    n = min(len(dataset), max_samples)

    for i in tqdm(range(n), desc="Predicting + sausage split (2-pass)"):
        image_t, mask_t = dataset[i]
        image_b = image_t.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(image_b)

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred_mask = (prob > float(thresh)).astype(np.uint8)
        img_vis = denormalize_imagenet(image_t)

        # ---- Pass 1: your current settings (thin ~1.5 px) ----
        labels1 = split_instances_geom_shape_rowaware(
            pred_mask,
            min_area=int(min_area), open_radius=int(open_radius),
            step_deg=10, jump_factor=1.45,
            radius_q=(0.35, 0.65), spacing_scale=0.98,
            cut_px=1.5, thin_cuts=True,
            valley_smooth=0.02,
            bins_min=int(bins_min), bins_max=int(bins_max),
        )

        labels = labels1

        # ---- Pass 2: more aggressive finish (on pass-1 mask) ----
        if second_pass:
            mask_after = (labels1 > 0).astype(np.uint8)
            labels2 = split_instances_geom_shape_rowaware(
                mask_after,
                min_area=int(min_area), open_radius=0,   # no extra opening now
                step_deg=int(sp_step_deg),
                jump_factor=float(sp_jump_factor),
                radius_q=tuple(sp_radius_q),
                spacing_scale=float(sp_spacing_scale),
                valley_smooth=float(sp_valley_smooth),
                bins_min=int(sp_bins_min), bins_max=int(sp_bins_max),
                cut_px=float(sp_cut_px), thin_cuts=bool(sp_thin_cuts),
            )
            labels = labels2

        n_instances = int(labels.max())
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)

        # overlays
        sem_overlay = img_vis.copy()
        sem_overlay[pred_mask.astype(bool)] = (
            0.5 * sem_overlay[pred_mask.astype(bool)] +
            0.5 * np.array([0, 255, 0], dtype=np.float32)
        ).astype(np.uint8)

        inst_overlay = overlay_instances_on_image(img_vis, labels, alpha=0.35)
        boundaries = _seg.find_boundaries(labels, mode="outer")
        boundary_overlay = img_vis.copy()
        boundary_overlay[boundaries] = [255, 0, 0]

        fig, axs = plt.subplots(1, 5, figsize=(20, 4))
        axs[0].imshow(img_vis);              axs[0].set_title("Image");                 axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray"); axs[1].set_title("Ground Truth");          axs[1].axis("off")
        axs[2].imshow(sem_overlay);          axs[2].set_title("Semantic Overlay");      axs[2].axis("off")
        axs[3].imshow(inst_overlay);         axs[3].set_title(f"Instances (N={n_instances})"); axs[3].axis("off")
        axs[4].imshow(boundary_overlay);     axs[4].set_title("Instance Boundaries");   axs[4].axis("off")
        plt.tight_layout(); plt.show()

        print(f"[{i+1}/{n}] separated instances: {n_instances}")


# =================== example call (2-pass enabled) ===================
visualize_predictions_with_instances_geom(
    best_model, test_dataset, device,
    max_samples=10, thresh=0.66,
    min_area=220, open_radius=1,
    # second-pass ON with aggressive knobs
    second_pass=True,
    sp_step_deg=10,
    sp_jump_factor=1.28,
    sp_radius_q=(0.25, 0.55),
    sp_spacing_scale=0.90,
    sp_valley_smooth=0.012,
    sp_bins_min=64, sp_bins_max=384,
    sp_cut_px=2.0, sp_thin_cuts=True,
)


In [ ]:
# ================== RADIAL-JUMPS ROW/COL "SAUSAGE" SPLITTER
# === (thin cuts + per-image radius mode + stronger 2nd pass + final micro-pass) ===
import numpy as np
from scipy import ndimage as ndi
from skimage import morphology, measure
from skimage.morphology import disk, thin

# ----------------------------- helpers -----------------------------
def _line_kernel(angle_rad, length=31, thickness=3):
    L = int(max(3, length)); W = int(max(1, thickness))
    size = int(np.ceil(L*np.sqrt(2))) + 2*W + 3
    k = np.zeros((size, size), np.uint8); c = size//2
    s = np.linspace(-(L-1)/2, (L-1)/2, L, dtype=np.float32)
    yy = c + s*np.sin(angle_rad); xx = c + s*np.cos(angle_rad)
    rr = np.clip(np.round(yy).astype(int), 0, size-1)
    cc = np.clip(np.round(xx).astype(int), 0, size-1)
    k[rr, cc] = 1
    if W > 1:
        k = morphology.binary_dilation(k, disk(W//2)).astype(np.uint8)
    return k.astype(bool)

def _radii_from_centroid(reg_sub, cy, cx, step_deg=15, max_step=None):
    """Sample radius from centroid to boundary at angles 0..π (step_deg)."""
    Hs, Ws = reg_sub.shape
    if max_step is None:
        max_step = np.hypot(Hs, Ws)
    deg = np.arange(0, 180, step_deg, dtype=np.float32)
    ang = np.deg2rad(deg)
    radii = np.zeros_like(ang, dtype=np.float32)

    for i, a in enumerate(ang):
        dx, dy = np.cos(a), np.sin(a)
        x, y = float(cx), float(cy)
        r = 0.0
        while 0 <= int(round(y)) < Hs and 0 <= int(round(x)) < Ws and reg_sub[int(round(y)), int(round(x))]:
            x += dx; y += dy; r += 1.0
            if r > max_step:
                break
        radii[i] = r
    return deg, ang, radii

def _valleys_1d(profile, k_cuts, smooth_sigma):
    """Pick k valley indices from a 1-D profile (cheap)."""
    if k_cuts <= 0 or profile.size < 8:
        return []
    p = ndi.gaussian_filter1d(profile.astype(np.float32), smooth_sigma, mode='nearest')
    inv = -p
    is_peak = (inv > np.r_[inv[1:], -np.inf]) & (inv > np.r_[-np.inf, inv[:-1]])
    idx = np.where(is_peak)[0]
    if idx.size == 0:
        return []
    order = np.argsort(p[idx])  # smallest p = deepest valley
    idx = idx[order[:k_cuts]]
    idx.sort()
    return idx.tolist()

def _equiv_radius_from_area(area):
    return float(np.sqrt(float(area) / np.pi))

def _image_radius_mode(labels, min_area=220):
    """Most common per-image crown radius (via FD histogram on equivalent radii)."""
    radii = [ _equiv_radius_from_area(rp.area)
              for rp in measure.regionprops(labels) if rp.area >= int(min_area) ]
    if not radii:
        return 3.0, (2.0, 4.0)
    r = np.asarray(radii, np.float32)
    r_min, r_max = float(r.min()), float(r.max())
    if r_max <= r_min + 1e-6:
        med = float(np.median(r));  return med, (r_min, r_max)
    iqr = float(np.percentile(r, 75) - np.percentile(r, 25))
    bw  = 2.0 * max(iqr, 1e-6) * (r.size ** (-1/3))  # Freedman–Diaconis
    if not np.isfinite(bw) or bw <= 0:
        bw = max(0.25*(r_max - r_min), 1.0)
    n_bins = int(np.clip(np.ceil((r_max - r_min)/bw), 5, 40))
    hist, edges = np.histogram(r, bins=n_bins, range=(r_min, r_max))
    k = int(np.argmax(hist))
    mask = (r >= edges[k]) & (r < edges[k+1])
    R_mode = float(np.median(r[mask])) if np.any(mask) else float(np.median(r))
    return R_mode, (float(edges[k]), float(edges[k+1]))

# ------------------- main splitter (first pass) --------------------
def split_instances_geom_shape_rowaware(
    pred_mask,
    image_rgb=None,
    # cleanup
    min_area=220, open_radius=1,
    # legacy/compat (accepted but unused)
    neck_rel=0.28, neck_dilate=4, neck_down=3,
    seed_radius=3, min_peak_distance=4, h_minima=1.1,
    gaussian_sigma=0.5, dist_gamma=1.25,
    compactness=5.0, edge_weight=0.36,
    row_cut_len=0, row_cut_thick=1,
    col_cut_len=0, col_cut_thick=1,
    # RADIAL-JUMPS core knobs
    step_deg=15,
    jump_factor=1.45,
    radius_q=(0.35, 0.65),
    spacing_scale=0.98,
    band_frac=0.10,          # used if cut_px is None
    valley_smooth=0.02,
    bins_min=48, bins_max=256,
    # thin cuts controller
    cut_px=None,             # exact cut thickness (px). If set, overrides band_frac
    thin_cuts=True,          # skeletonize band to ~1 px, then redilate to cut_px
    min_cut_px=1.0,          # never thinner than 1 px total
    # compatibility aliases
    ar_trigger=None, band_frac_row=None, band_frac_col=None,
    # swallow extra kwargs safely
    **kwargs
):
    # map legacy row/col frac to a single frac (if user passes those)
    if (band_frac_row is not None) or (band_frac_col is not None):
        vals = [v for v in (band_frac_row, band_frac_col) if v is not None]
        if vals: band_frac = float(np.mean(vals))

    # helper → compute half-thickness
    def _half(span, cut_px, band_frac, min_cut_px):
        if cut_px is not None:
            return max(min_cut_px/2.0, float(cut_px)/2.0)
        return max(min_cut_px/2.0, float(band_frac) * float(span))

    # ---- pre-clean ----
    m = pred_mask.astype(bool)
    if open_radius and open_radius > 0:
        m = morphology.opening(m, disk(int(open_radius)))
    m = morphology.remove_small_objects(m, min_size=int(min_area))
    if not m.any():
        return np.zeros_like(pred_mask, np.int32)

    labeled = measure.label(m, connectivity=1)

    # ----- first pass: per-object base radius & jump directions -----
    base_radii, jump_angles, obj_info = [], [], []

    for r in measure.regionprops(labeled):
        if r.area < min_area:
            continue
        minr, minc, maxr, maxc = r.bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == r.label)

        cy, cx = r.centroid
        cy -= minr; cx -= minc

        deg, ang, radii = _radii_from_centroid(reg_sub, cy, cx, step_deg=step_deg)

        lo, hi = np.quantile(radii, radius_q)
        base = 0.5*(lo + hi)
        base_radii.append(base)

        jump_idx = np.where(radii > (jump_factor * base))[0]
        if jump_idx.size:
            jump_angles.extend(deg[jump_idx].tolist())

        obj_info.append((r.label, (minr, minc, maxr, maxc), (cy, cx), deg, ang, radii))

    # global radius estimate (robust)
    R_est = float(np.median(base_radii)) if base_radii else 3.0
    R_est = max(1.0, R_est)

    # global row/col angles (via jump histogram; fallback = tensor-like)
    if len(jump_angles) >= 3:
        hist, edges = np.histogram(jump_angles, bins=36, range=(0,180))
        peak_deg = float(0.5*(edges[np.argmax(hist)] + edges[np.argmax(hist)+1]))
        ang_row = np.deg2rad(peak_deg)
    else:
        bg = (~m).astype(np.float32)
        g  = ndi.gaussian_filter(bg, 2.0)
        gy, gx = np.gradient(g)
        Jxx = ndi.gaussian_filter(gx*gx, 2.0)
        Jxy = ndi.gaussian_filter(gx*gy, 2.0)
        Jyy = ndi.gaussian_filter(gy*gy, 2.0)
        theta = 0.5*np.arctan2(2*Jxy, (Jxx - Jyy + 1e-8))
        vals = theta[bg > np.percentile(bg, 50)]
        ang_row = float(np.median(vals)) if vals.size else 0.0

    ang_col = (ang_row + np.pi/2.0) % np.pi
    cos_r, sin_r = np.cos(ang_row), np.sin(ang_row)
    cos_c, sin_c = np.cos(ang_col), np.sin(ang_col)

    # ----- second stage (within first pass): cut elongated blobs along row/col -----
    cut_global = np.zeros_like(m, bool)

    for (lbl, bbox, (cy, cx), deg, ang, radii) in obj_info:
        minr, minc, maxr, maxc = bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == lbl)
        Hs, Ws = reg_sub.shape

        yy, xx = np.mgrid[0:Hs, 0:Ws]
        yy_abs, xx_abs = yy + minr, xx + minc

        # row axis
        t_row = xx_abs * cos_r + yy_abs * sin_r
        s_col = -xx_abs * sin_r + yy_abs * cos_r
        t_vals = t_row[reg_sub]; s_vals = s_col[reg_sub]
        t_min, t_max = float(t_vals.min()), float(t_vals.max())
        s_min, s_max = float(s_vals.min()), float(s_vals.max())
        Lr = t_max - t_min; Wc = s_max - s_min + 1e-6

        # column axis
        t_col = xx_abs * cos_c + yy_abs * sin_c
        s_row = -xx_abs * sin_c + yy_abs * cos_c
        tc_vals = t_col[reg_sub]; sr_vals = s_row[reg_sub]
        tc_min, tc_max = float(tc_vals.min()), float(tc_vals.max())
        sr_min, sr_max = float(sr_vals.min()), float(sr_vals.max())
        Lc = tc_max - tc_min; Wr = sr_max - sr_min + 1e-6

        D = 2.0 * R_est * float(spacing_scale)
        n_row = int(np.round(Lr / max(D, 1.0)))
        n_col = int(np.round(Lc / max(D, 1.0)))

        cut_local = np.zeros_like(reg_sub, bool)

        # A) along rows
        if n_row >= 2 and (Lr / Wc) >= 1.2:
            nb = int(np.clip(np.round(Lr), bins_min, bins_max))
            idx = np.clip(((t_row - t_min) / (Lr + 1e-6) * nb).astype(int), 0, nb-1)
            prof = np.bincount(idx[reg_sub], minlength=nb)
            k = max(1, n_row - 1)
            valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
            if valleys:
                t_bounds = [t_min + (vi + 0.5) * (Lr / nb) for vi in valleys]
                half = _half(Wc, cut_px, band_frac, min_cut_px)
                for tb in t_bounds:
                    cut_local |= (np.abs(t_row - tb) <= half)

        # B) along columns
        if n_col >= 2 and (Lc / Wr) >= 1.2:
            nb = int(np.clip(np.round(Lc), bins_min, bins_max))
            idx = np.clip(((t_col - tc_min) / (Lc + 1e-6) * nb).astype(int), 0, nb-1)
            prof = np.bincount(idx[reg_sub], minlength=nb)
            k = max(1, n_col - 1)
            valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
            if valleys:
                t_bounds = [tc_min + (vi + 0.5) * (Lc / nb) for vi in valleys]
                half = _half(Wr, cut_px, band_frac, min_cut_px)
                for tb in t_bounds:
                    cut_local |= (np.abs(t_col - tb) <= half)

        # thin the cut bands if requested
        if thin_cuts and cut_local.any():
            cut_local = thin(cut_local)  # ~1 px
            if (cut_px is not None) and (cut_px > 1.0):
                rad = int(max(0, np.round(cut_px/2.0) - 1))
                if rad > 0:
                    cut_local = morphology.binary_dilation(cut_local, disk(rad))

        cut_global[minr:maxr, minc:maxc] |= (cut_local & reg_sub)

    # apply cuts + relabel
    if cut_global.any():
        m = m & (~cut_global)

    m = morphology.remove_small_objects(m, min_size=int(min_area))
    labels = measure.label(m, connectivity=1).astype(np.int32)
    return labels

# ---------------- second pass (radius-mode–driven + micro-pass) ----------------
def _second_pass_by_radius_general(
    mask_after,
    R_mode,
    min_area=220,
    # target segments: k_area ≈ (R_eq/R_mode)^2 * k_boost
    k_boost=1.0,
    # anisotropy boost
    ar_floor=1.25,
    ar_gain=0.55,
    max_k=14,
    # splitter knobs for re-splitting
    step_deg=8,
    jump_factor=1.25,
    radius_q=(0.22, 0.52),
    spacing_scale=0.86,
    valley_smooth=0.010,
    bins_min=64, bins_max=512,
    cut_px=1.7, thin_cuts=True,
    # final micro-pass on oversized pieces
    final_oversize_factor=1.6,   # if R_eq/R_mode >= this, try one more split
    final_step_deg=6,
    final_jump_factor=1.20,
    final_radius_q=(0.20, 0.50),
    final_spacing_scale=0.82,
    final_valley_smooth=0.009,
    final_cut_px=1.5
):
    lbl = measure.label(mask_after.astype(bool), connectivity=1)
    out = np.zeros_like(lbl, np.int32); next_id = 1

    for rp in measure.regionprops(lbl):
        if rp.area < int(min_area):
            continue

        # target k from area + anisotropy
        R_eq   = _equiv_radius_from_area(rp.area)
        k_area = (R_eq / max(R_mode, 1e-6)) ** 2
        k_area = int(np.clip(np.round(k_boost * k_area), 1, max_k))

        AR = (float(rp.major_axis_length) /
              max(float(rp.minor_axis_length), 1e-6)) if rp.minor_axis_length > 0 else 1.0
        k_aniso = 1 + int(max(0.0, ar_gain * max(0.0, AR - ar_floor)))

        k_target = max(k_area, k_aniso)

        # submask and re-split
        minr, minc, maxr, maxc = rp.bbox
        sub = (lbl[minr:maxr, minc:maxc] == rp.label).astype(np.uint8)

        sub_labels = split_instances_geom_shape_rowaware(
            sub,
            min_area=int(min_area), open_radius=0,
            step_deg=int(step_deg),
            jump_factor=float(jump_factor),
            radius_q=tuple(radius_q),
            spacing_scale=float(spacing_scale),
            valley_smooth=float(valley_smooth),
            bins_min=int(bins_min), bins_max=int(bins_max),
            cut_px=float(cut_px), thin_cuts=bool(thin_cuts),
        )

        # one more stronger try if not enough pieces
        if sub_labels.max() < k_target:
            sub_labels = split_instances_geom_shape_rowaware(
                sub,
                min_area=int(min_area), open_radius=0,
                step_deg=int(final_step_deg),
                jump_factor=float(final_jump_factor),
                radius_q=tuple(final_radius_q),
                spacing_scale=float(final_spacing_scale),
                valley_smooth=float(final_valley_smooth),
                bins_min=int(bins_min), bins_max=int(bins_max),
                cut_px=float(final_cut_px), thin_cuts=bool(thin_cuts),
            )

        # micro-pass on any oversized parts after splitting
        stitched = np.zeros_like(sub_labels)
        nxt = 1
        for j in range(1, sub_labels.max()+1):
            part = (sub_labels == j)
            if part.sum() < int(min_area):
                continue
            # check oversize
            rp2 = measure.regionprops(part.astype(np.uint8))[0]
            R_eq2 = _equiv_radius_from_area(rp2.area)
            if (R_eq2 / max(R_mode, 1e-6)) >= float(final_oversize_factor):
                # try a final micro-split on this part
                part_labels = split_instances_geom_shape_rowaware(
                    part.astype(np.uint8),
                    min_area=int(min_area), open_radius=0,
                    step_deg=int(final_step_deg),
                    jump_factor=float(final_jump_factor),
                    radius_q=tuple(final_radius_q),
                    spacing_scale=float(final_spacing_scale),
                    valley_smooth=float(final_valley_smooth),
                    bins_min=int(bins_min), bins_max=int(bins_max),
                    cut_px=float(final_cut_px), thin_cuts=bool(thin_cuts),
                )
                if part_labels.max() == 0:
                    stitched[part] = nxt; nxt += 1
                else:
                    for k in range(1, part_labels.max()+1):
                        piece = (part_labels == k)
                        if piece.sum() < int(min_area):
                            continue
                        stitched[piece] = nxt; nxt += 1
            else:
                stitched[part] = nxt; nxt += 1

        if stitched.max() == 0:
            out[minr:maxr, minc:maxc][sub.astype(bool)] = next_id; next_id += 1
        else:
            for j in range(1, stitched.max()+1):
                piece = (stitched == j)
                if piece.sum() < int(min_area): continue
                out[minr:maxr, minc:maxc][piece] = next_id; next_id += 1

    out = morphology.remove_small_objects(out, min_size=int(min_area))
    out = measure.label(out > 0, connectivity=1).astype(np.int32)
    return out

# ----------------------------- visualization -----------------------------
def overlay_instances_on_image(image_rgb_uint8, labels, alpha=0.35):
    from skimage import color
    img_f = np.clip(image_rgb_uint8.astype(np.float32) / 255.0, 0, 1)
    vis = color.label2rgb(labels, image=img_f, bg_label=0, alpha=float(alpha))
    return (np.clip(vis, 0, 1) * 255).astype(np.uint8)

def visualize_predictions_with_instances_geom(
    model, dataset, device,
    max_samples=15, thresh=0.65,
    # legacy compat knobs (unused by the splitter but kept for API stability)
    min_area=220, open_radius=1,
    neck_rel=0.25, neck_dilate=4, neck_down=3,
    seed_radius=3, min_peak_distance=4,
    gaussian_sigma=0.5, dist_gamma=1.25,
    compactness=5.0, edge_weight=0.36,
    row_cut_len=0, row_cut_thick=1,
    col_cut_len=0, col_cut_thick=1,
    h_minima=1.0,
    # first-pass bins
    bins_min=48, bins_max=256,
    # second-pass toggle & knobs (general, radius-mode driven)
    second_pass=True,
    sp_k_boost=1.35,          # ↑ more splitting
    sp_ar_floor=1.20,
    sp_ar_gain=0.60,
    sp_max_k=14,
    sp_step_deg=8,
    sp_jump_factor=1.25,
    sp_radius_q=(0.22, 0.52),
    sp_spacing_scale=0.86,
    sp_valley_smooth=0.010,
    sp_bins_min=64, sp_bins_max=512,
    sp_cut_px=1.7, sp_thin_cuts=True,
    sp_final_oversize=1.60,   # micro-pass trigger: R_eq/R_mode ≥ this
):
    import torch, matplotlib.pyplot as plt
    from tqdm import tqdm
    from skimage import segmentation as _seg

    model.eval()
    n = min(len(dataset), max_samples)

    for i in tqdm(range(n), desc="Predicting + sausage split (more aggressive 2-pass)"):
        image_t, mask_t = dataset[i]
        image_b = image_t.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(image_b)

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred_mask = (prob > float(thresh)).astype(np.uint8)
        img_vis = denormalize_imagenet(image_t)

        # ---- Pass 1: slightly more aggressive than before ----
        labels1 = split_instances_geom_shape_rowaware(
            pred_mask,
            min_area=int(min_area), open_radius=int(open_radius),
            step_deg=8,                     # ↓ finer radial sampling
            jump_factor=1.35,               # ↓ easier to flag merges
            radius_q=(0.32, 0.62),
            spacing_scale=0.94,             # ↑ predicts more trees along axes
            cut_px=1.3, thin_cuts=True,     # thin visual cuts
            valley_smooth=0.015,
            bins_min=int(bins_min), bins_max=int(bins_max),
        )

        labels = labels1

        # ---- Per-image radius mode & stronger second pass ----
        if second_pass:
            R_mode, _ = _image_radius_mode(labels1, min_area=int(min_area))
            mask_after = (labels1 > 0).astype(np.uint8)
            labels2 = _second_pass_by_radius_general(
                mask_after,
                R_mode=R_mode,
                min_area=int(min_area),
                k_boost=float(sp_k_boost),
                ar_floor=float(sp_ar_floor),
                ar_gain=float(sp_ar_gain),
                max_k=int(sp_max_k),
                step_deg=int(sp_step_deg),
                jump_factor=float(sp_jump_factor),
                radius_q=tuple(sp_radius_q),
                spacing_scale=float(sp_spacing_scale),
                valley_smooth=float(sp_valley_smooth),
                bins_min=int(sp_bins_min), bins_max=int(sp_bins_max),
                cut_px=float(sp_cut_px), thin_cuts=bool(sp_thin_cuts),
                final_oversize_factor=float(sp_final_oversize),
            )
            labels = labels2

        n_instances = int(labels.max())
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)

        # overlays
        sem_overlay = img_vis.copy()
        m = pred_mask.astype(bool)
        sem_overlay[m] = (0.5 * sem_overlay[m] + 0.5 * np.array([0, 255, 0], dtype=np.float32)).astype(np.uint8)

        inst_overlay = overlay_instances_on_image(img_vis, labels, alpha=0.35)
        boundaries = _seg.find_boundaries(labels, mode="outer")
        boundary_overlay = img_vis.copy()
        boundary_overlay[boundaries] = [255, 0, 0]

        fig, axs = plt.subplots(1, 5, figsize=(20, 4))
        axs[0].imshow(img_vis);              axs[0].set_title("Image");                 axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray"); axs[1].set_title("Ground Truth");          axs[1].axis("off")
        axs[2].imshow(sem_overlay);          axs[2].set_title("Semantic Overlay");      axs[2].axis("off")
        axs[3].imshow(inst_overlay);         axs[3].set_title(f"Instances (N={n_instances})"); axs[3].axis("off")
        axs[4].imshow(boundary_overlay);     axs[4].set_title("Instance Boundaries");   axs[4].axis("off")
        plt.tight_layout(); plt.show()

        print(f"[{i+1}/{n}] separated instances: {n_instances}")


# =================== example call (more aggressive) ===================
visualize_predictions_with_instances_geom(
    best_model, test_dataset, device,
    max_samples=10, thresh=0.66,
    min_area=220, open_radius=1,
    second_pass=True,
    sp_k_boost=1.35,
    sp_ar_floor=1.20,
    sp_ar_gain=0.60,
    sp_max_k=14,
    sp_step_deg=8,
    sp_jump_factor=1.25,
    sp_radius_q=(0.22, 0.52),
    sp_spacing_scale=0.86,
    sp_valley_smooth=0.010,
    sp_bins_min=64, sp_bins_max=512,
    sp_cut_px=1.7, sp_thin_cuts=True,
    sp_final_oversize=1.60,
)


In [ ]:
# ================== RADIAL-JUMPS ROW/COL "SAUSAGE" SPLITTER (x4 same-method passes) ==================
# All 4 intervals use the SAME row/col radial-jumps method.
# Each subsequent interval is just a stronger re-run on the current mask (no Gaussian gates).

import numpy as np
from scipy import ndimage as ndi
from skimage import morphology, measure
from skimage.morphology import disk, thin

# ----------------------------- helpers -----------------------------
def _line_kernel(angle_rad, length=31, thickness=3):
    L = int(max(3, length)); W = int(max(1, thickness))
    size = int(np.ceil(L*np.sqrt(2))) + 2*W + 3
    k = np.zeros((size, size), np.uint8); c = size//2
    s = np.linspace(-(L-1)/2, (L-1)/2, L, dtype=np.float32)
    yy = c + s*np.sin(angle_rad); xx = c + s*np.cos(angle_rad)
    rr = np.clip(np.round(yy).astype(int), 0, size-1)
    cc = np.clip(np.round(xx).astype(int), 0, size-1)
    k[rr, cc] = 1
    if W > 1:
        k = morphology.binary_dilation(k, disk(W//2)).astype(np.uint8)
    return k.astype(bool)

def _radii_from_centroid(reg_sub, cy, cx, step_deg=15, max_step=None):
    """Sample radius from centroid to boundary at angles 0..π (step_deg)."""
    Hs, Ws = reg_sub.shape
    if max_step is None:
        max_step = np.hypot(Hs, Ws)
    deg = np.arange(0, 180, step_deg, dtype=np.float32)
    ang = np.deg2rad(deg)
    radii = np.zeros_like(ang, dtype=np.float32)

    for i, a in enumerate(ang):
        dx, dy = np.cos(a), np.sin(a)
        x, y = float(cx), float(cy)
        r = 0.0
        while 0 <= int(round(y)) < Hs and 0 <= int(round(x)) < Ws and reg_sub[int(round(y)), int(round(x))]:
            x += dx; y += dy; r += 1.0
            if r > max_step:
                break
        radii[i] = r
    return deg, ang, radii

def _valleys_1d(profile, k_cuts, smooth_sigma):
    """Pick k valley indices from a 1-D profile (cheap)."""
    if k_cuts <= 0 or profile.size < 8:
        return []
    p = ndi.gaussian_filter1d(profile.astype(np.float32), smooth_sigma, mode='nearest')
    inv = -p
    is_peak = (inv > np.r_[inv[1:], -np.inf]) & (inv > np.r_[-np.inf, inv[:-1]])
    idx = np.where(is_peak)[0]
    if idx.size == 0:
        return []
    order = np.argsort(p[idx])  # smallest p = deepest valley
    idx = idx[order[:k_cuts]]
    idx.sort()
    return idx.tolist()

def _equiv_radius_from_area(area):
    return float(np.sqrt(float(area) / np.pi))

def _image_radius_mode(labels, min_area=220):
    """Most common per-image crown radius (via FD histogram on equivalent radii)."""
    radii = [ _equiv_radius_from_area(rp.area)
              for rp in measure.regionprops(labels) if rp.area >= int(min_area) ]
    if not radii:
        return 3.0, (2.0, 4.0)
    r = np.asarray(radii, np.float32)
    r_min, r_max = float(r.min()), float(r.max())
    if r_max <= r_min + 1e-6:
        med = float(np.median(r));  return med, (r_min, r_max)
    iqr = float(np.percentile(r, 75) - np.percentile(r, 25))
    bw  = 2.0 * max(iqr, 1e-6) * (r.size ** (-1/3))  # Freedman–Diaconis
    if not np.isfinite(bw) or bw <= 0:
        bw = max(0.25*(r_max - r_min), 1.0)
    n_bins = int(np.clip(np.ceil((r_max - r_min)/bw), 5, 40))
    hist, edges = np.histogram(r, bins=n_bins, range=(r_min, r_max))
    k = int(np.argmax(hist))
    mask = (r >= edges[k]) & (r < edges[k+1])
    R_mode = float(np.median(r[mask])) if np.any(mask) else float(np.median(r))
    return R_mode, (float(edges[k]), float(edges[k+1]))

# ------------------- main splitter (ROW/COL; supports slight angle jitter) --------------------
def split_instances_geom_shape_rowaware(
    pred_mask,
    image_rgb=None,
    # cleanup
    min_area=220, open_radius=1,
    # legacy/compat (accepted but unused)
    neck_rel=0.28, neck_dilate=4, neck_down=3,
    seed_radius=3, min_peak_distance=4, h_minima=1.1,
    gaussian_sigma=0.5, dist_gamma=1.25,
    compactness=5.0, edge_weight=0.36,
    row_cut_len=0, row_cut_thick=1,
    col_cut_len=0, col_cut_thick=1,
    # RADIAL-JUMPS core knobs
    step_deg=15,
    jump_factor=1.45,
    radius_q=(0.35, 0.65),
    spacing_scale=0.98,
    band_frac=0.10,          # used if cut_px is None
    valley_smooth=0.02,
    bins_min=48, bins_max=256,
    # thin cuts controller
    cut_px=None,             # exact cut thickness (px). If set, overrides band_frac
    thin_cuts=True,          # skeletonize band to ~1 px, then redilate to cut_px
    min_cut_px=1.0,          # never thinner than 1 px total
    # orientation controls (keep SAME method; just allow tiny angle jitter)
    angle_override_deg=None,  # if given, force this base angle (deg) for row-axis
    angle_jitter_deg=0.0,     # try base±jitter (deg) and union the cuts (0 = off)
    # swallow extra kwargs safely
    **kwargs
):
    # map legacy row/col frac to a single frac (if user passes those)
    band_frac_row = kwargs.get("band_frac_row", None)
    band_frac_col = kwargs.get("band_frac_col", None)
    if (band_frac_row is not None) or (band_frac_col is not None):
        vals = [v for v in (band_frac_row, band_frac_col) if v is not None]
        if vals: band_frac = float(np.mean(vals))

    # helper → compute half-thickness
    def _half(span, cut_px, band_frac, min_cut_px):
        if cut_px is not None:
            return max(min_cut_px/2.0, float(cut_px)/2.0)
        return max(min_cut_px/2.0, float(band_frac) * float(span))

    # ---- pre-clean ----
    m = pred_mask.astype(bool)
    if open_radius and open_radius > 0:
        m = morphology.opening(m, disk(int(open_radius)))
    m = morphology.remove_small_objects(m, min_size=int(min_area))
    if not m.any():
        return np.zeros_like(pred_mask, np.int32)

    labeled = measure.label(m, connectivity=1)

    # ----- first pass: per-object base radius & jump directions -----
    base_radii, jump_angles, obj_info = [], [], []

    for r in measure.regionprops(labeled):
        if r.area < min_area:
            continue
        minr, minc, maxr, maxc = r.bbox
        reg_sub = (labeled[minr:maxr, minc:maxc] == r.label)
        cy, cx = r.centroid; cy -= minr; cx -= minc

        deg, ang, radii = _radii_from_centroid(reg_sub, cy, cx, step_deg=step_deg)

        lo, hi = np.quantile(radii, radius_q)
        base = 0.5*(lo + hi)
        base_radii.append(base)

        jump_idx = np.where(radii > (jump_factor * base))[0]
        if jump_idx.size:
            jump_angles.extend(deg[jump_idx].tolist())

        obj_info.append((r.label, (minr, minc, maxr, maxc), (cy, cx), deg, ang, radii))

    # global radius estimate (robust)
    R_est = float(np.median(base_radii)) if base_radii else 3.0
    R_est = max(1.0, R_est)

    # global row angle
    if angle_override_deg is not None:
        ang_row_base = (np.deg2rad(float(angle_override_deg)) % np.pi)
    elif len(jump_angles) >= 3:
        hist, edges = np.histogram(jump_angles, bins=36, range=(0,180))
        peak_deg = float(0.5*(edges[np.argmax(hist)] + edges[np.argmax(hist)+1]))
        ang_row_base = np.deg2rad(peak_deg)
    else:
        bg = (~m).astype(np.float32)
        g  = ndi.gaussian_filter(bg, 2.0)
        gy, gx = np.gradient(g)
        Jxx = ndi.gaussian_filter(gx*gx, 2.0)
        Jxy = ndi.gaussian_filter(gx*gy, 2.0)
        Jyy = ndi.gaussian_filter(gy*gy, 2.0)
        theta = 0.5*np.arctan2(2*Jxy, (Jxx - Jyy + 1e-8))
        vals = theta[bg > np.percentile(bg, 50)]
        ang_row_base = float(np.median(vals)) if vals.size else 0.0

    # optional tiny jitter to better catch near-diagonals without changing method
    jitter_list = [0.0]
    if angle_jitter_deg and float(angle_jitter_deg) > 0:
        d = np.deg2rad(float(angle_jitter_deg))
        jitter_list = [-d, 0.0, +d]

    cut_global = np.zeros_like(m, bool)

    for jitter in jitter_list:
        ang_row = (ang_row_base + jitter) % np.pi
        ang_col = (ang_row + np.pi/2.0) % np.pi
        cos_r, sin_r = np.cos(ang_row), np.sin(ang_row)
        cos_c, sin_c = np.cos(ang_col), np.sin(ang_col)

        for (lbl, bbox, (cy, cx), deg, ang, radii) in obj_info:
            minr, minc, maxr, maxc = bbox
            reg_sub = (labeled[minr:maxr, minc:maxc] == lbl)
            Hs, Ws = reg_sub.shape

            yy, xx = np.mgrid[0:Hs, 0:Ws]
            yy_abs, xx_abs = yy + minr, xx + minc

            # row axis
            t_row = xx_abs * cos_r + yy_abs * sin_r
            s_col = -xx_abs * sin_r + yy_abs * cos_r
            t_vals = t_row[reg_sub]; s_vals = s_col[reg_sub]
            t_min, t_max = float(t_vals.min()), float(t_vals.max())
            s_min, s_max = float(s_vals.min()), float(s_vals.max())
            Lr = t_max - t_min; Wc = s_max - s_min + 1e-6

            # column axis
            t_col = xx_abs * cos_c + yy_abs * sin_c
            s_row = -xx_abs * sin_c + yy_abs * cos_c
            tc_vals = t_col[reg_sub]; sr_vals = s_row[reg_sub]
            tc_min, tc_max = float(tc_vals.min()), float(tc_vals.max())
            sr_min, sr_max = float(sr_vals.min()), float(sr_vals.max())
            Lc = tc_max - tc_min; Wr = sr_max - sr_min + 1e-6

            D = 2.0 * R_est * float(spacing_scale)
            n_row = int(np.round(Lr / max(D, 1.0)))
            n_col = int(np.round(Lc / max(D, 1.0)))

            cut_local = np.zeros_like(reg_sub, bool)

            # A) along rows
            if n_row >= 2 and (Lr / Wc) >= 1.2:
                nb = int(np.clip(np.round(Lr), bins_min, bins_max))
                idx = np.clip(((t_row - t_min) / (Lr + 1e-6) * nb).astype(int), 0, nb-1)
                prof = np.bincount(idx[reg_sub], minlength=nb)
                k = max(1, n_row - 1)
                valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
                if valleys:
                    t_bounds = [t_min + (vi + 0.5) * (Lr / nb) for vi in valleys]
                    half = _half(Wc, cut_px, band_frac, min_cut_px)
                    for tb in t_bounds:
                        cut_local |= (np.abs(t_row - tb) <= half)

            # B) along columns
            if n_col >= 2 and (Lc / Wr) >= 1.2:
                nb = int(np.clip(np.round(Lc), bins_min, bins_max))
                idx = np.clip(((t_col - tc_min) / (Lc + 1e-6) * nb).astype(int), 0, nb-1)
                prof = np.bincount(idx[reg_sub], minlength=nb)
                k = max(1, n_col - 1)
                valleys = _valleys_1d(prof, k, smooth_sigma=max(1.0, valley_smooth*nb))
                if valleys:
                    t_bounds = [tc_min + (vi + 0.5) * (Lc / nb) for vi in valleys]
                    half = _half(Wr, cut_px, band_frac, min_cut_px)
                    for tb in t_bounds:
                        cut_local |= (np.abs(t_col - tb) <= half)

            # thin the cut bands if requested
            if thin_cuts and cut_local.any():
                cut_local = thin(cut_local)  # ~1 px
                if (cut_px is not None) and (cut_px > 1.0):
                    rad = int(max(0, np.round(cut_px/2.0) - 1))
                    if rad > 0:
                        cut_local = morphology.binary_dilation(cut_local, disk(rad))

            cut_global[minr:maxr, minc:maxc] |= (cut_local & reg_sub)

    # apply cuts + relabel
    if cut_global.any():
        m = m & (~cut_global)

    m = morphology.remove_small_objects(m, min_size=int(min_area))
    labels = measure.label(m, connectivity=1).astype(np.int32)
    return labels

# ----------------------------- visualization (4 intervals, SAME method) -----------------------------
def overlay_instances_on_image(image_rgb_uint8, labels, alpha=0.35):
    from skimage import color
    img_f = np.clip(image_rgb_uint8.astype(np.float32) / 255.0, 0, 1)
    vis = color.label2rgb(labels, image=img_f, bg_label=0, alpha=float(alpha))
    return (np.clip(vis, 0, 1) * 255).astype(np.uint8)

def visualize_predictions_with_instances_geom(
    model, dataset, device,
    max_samples=15, thresh=0.65,
    # cleanup
    min_area=220, open_radius=1,
    # base bin limits
    bins_min=48, bins_max=256,
    # PASS-1 (baseline)
    p1_step_deg=10, p1_jump_factor=1.45, p1_radius_q=(0.35, 0.65),
    p1_spacing_scale=0.98, p1_valley_smooth=0.020, p1_cut_px=1.5, p1_angle_jitter_deg=0.0,
    # PASS-2 (stronger)
    p2_step_deg=8,  p2_jump_factor=1.35, p2_radius_q=(0.32, 0.62),
    p2_spacing_scale=0.94, p2_valley_smooth=0.016, p2_cut_px=1.6, p2_angle_jitter_deg=4.0,
    # PASS-3 (stronger+)
    p3_step_deg=8,  p3_jump_factor=1.28, p3_radius_q=(0.30, 0.60),
    p3_spacing_scale=0.90, p3_valley_smooth=0.013, p3_cut_px=1.8, p3_angle_jitter_deg=6.0,
    # PASS-4 (max)
    p4_step_deg=6,  p4_jump_factor=1.22, p4_radius_q=(0.28, 0.58),
    p4_spacing_scale=0.86, p4_valley_smooth=0.011, p4_cut_px=2.0, p4_angle_jitter_deg=8.0,
):
    import torch, matplotlib.pyplot as plt
    from tqdm import tqdm
    from skimage import segmentation as _seg

    model.eval()
    n = min(len(dataset), max_samples)

    for i in tqdm(range(n), desc="Predicting + 4x SAME row/col splitter (increasing strength)"):
        image_t, mask_t = dataset[i]
        image_b = image_t.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(image_b)

        prob = torch.sigmoid(logits)[0, 0].detach().cpu().numpy()
        pred_mask = (prob > float(thresh)).astype(np.uint8)
        img_vis = denormalize_imagenet(image_t)

        # ---- INTERVAL 1 ----
        labels1 = split_instances_geom_shape_rowaware(
            pred_mask,
            min_area=int(min_area), open_radius=int(open_radius),
            step_deg=int(p1_step_deg), jump_factor=float(p1_jump_factor),
            radius_q=tuple(p1_radius_q), spacing_scale=float(p1_spacing_scale),
            cut_px=float(p1_cut_px), thin_cuts=True,
            valley_smooth=float(p1_valley_smooth),
            bins_min=int(bins_min), bins_max=int(bins_max),
            angle_jitter_deg=float(p1_angle_jitter_deg),
        )

        # ---- INTERVAL 2 ----
        labels2 = split_instances_geom_shape_rowaware(
            (labels1 > 0).astype(np.uint8),
            min_area=int(min_area), open_radius=0,
            step_deg=int(p2_step_deg), jump_factor=float(p2_jump_factor),
            radius_q=tuple(p2_radius_q), spacing_scale=float(p2_spacing_scale),
            cut_px=float(p2_cut_px), thin_cuts=True,
            valley_smooth=float(p2_valley_smooth),
            bins_min=int(max(bins_min, 64)), bins_max=int(max(384, bins_max)),
            angle_jitter_deg=float(p2_angle_jitter_deg),
        )

        # ---- INTERVAL 3 ----
        labels3 = split_instances_geom_shape_rowaware(
            (labels2 > 0).astype(np.uint8),
            min_area=int(min_area), open_radius=0,
            step_deg=int(p3_step_deg), jump_factor=float(p3_jump_factor),
            radius_q=tuple(p3_radius_q), spacing_scale=float(p3_spacing_scale),
            cut_px=float(p3_cut_px), thin_cuts=True,
            valley_smooth=float(p3_valley_smooth),
            bins_min=int(max(bins_min, 64)), bins_max=int(512),
            angle_jitter_deg=float(p3_angle_jitter_deg),
        )

        # ---- INTERVAL 4 ----
        labels4 = split_instances_geom_shape_rowaware(
            (labels3 > 0).astype(np.uint8),
            min_area=int(min_area), open_radius=0,
            step_deg=int(p4_step_deg), jump_factor=float(p4_jump_factor),
            radius_q=tuple(p4_radius_q), spacing_scale=float(p4_spacing_scale),
            cut_px=float(p4_cut_px), thin_cuts=True,
            valley_smooth=float(p4_valley_smooth),
            bins_min=int(max(bins_min, 64)), bins_max=int(512),
            angle_jitter_deg=float(p4_angle_jitter_deg),
        )

        labels = labels4
        n_instances = int(labels.max())
        gt_mask = mask_t.squeeze(0).cpu().numpy().astype(np.uint8)

        # overlays
        sem_overlay = img_vis.copy()
        m = pred_mask.astype(bool)
        sem_overlay[m] = (0.5 * sem_overlay[m] + 0.5 * np.array([0, 255, 0], dtype=np.float32)).astype(np.uint8)

        inst_overlay = overlay_instances_on_image(img_vis, labels, alpha=0.35)
        boundaries = _seg.find_boundaries(labels, mode="outer")
        boundary_overlay = img_vis.copy()
        boundary_overlay[boundaries] = [255, 0, 0]

        fig, axs = plt.subplots(1, 5, figsize=(20, 4))
        axs[0].imshow(img_vis);              axs[0].set_title("Image");                 axs[0].axis("off")
        axs[1].imshow(gt_mask, cmap="gray"); axs[1].set_title("Ground Truth");          axs[1].axis("off")
        axs[2].imshow(sem_overlay);          axs[2].set_title("Semantic Overlay");      axs[2].axis("off")
        axs[3].imshow(inst_overlay);         axs[3].set_title(f"Instances (N={n_instances})"); axs[3].axis("off")
        axs[4].imshow(boundary_overlay);     axs[4].set_title("Instance Boundaries");   axs[4].axis("off")
        plt.tight_layout(); plt.show()

        print(f"[{i+1}/{n}] separated instances: {n_instances}")

# =================== example call (4x same-method passes, stronger each time) ===================
visualize_predictions_with_instances_geom(
    best_model, test_dataset, device,
    max_samples=10, thresh=0.66,
    min_area=220, open_radius=1,
    # P1 (stronger than before)
    p1_step_deg=8,   p1_jump_factor=1.40, p1_radius_q=(0.33, 0.63),
    p1_spacing_scale=0.96, p1_valley_smooth=0.018, p1_cut_px=1.6, p1_angle_jitter_deg=2.0,
    # P2 (stronger)
    p2_step_deg=6,   p2_jump_factor=1.30, p2_radius_q=(0.31, 0.61),
    p2_spacing_scale=0.92, p2_valley_smooth=0.014, p2_cut_px=1.8, p2_angle_jitter_deg=6.0,
    # P3 (stronger+)
    p3_step_deg=6,   p3_jump_factor=1.24, p3_radius_q=(0.28, 0.58),
    p3_spacing_scale=0.88, p3_valley_smooth=0.011, p3_cut_px=2.0, p3_angle_jitter_deg=10.0,
    # P4 (max aggressive)
    p4_step_deg=5,   p4_jump_factor=1.18, p4_radius_q=(0.26, 0.56),
    p4_spacing_scale=0.84, p4_valley_smooth=0.009, p4_cut_px=2.2, p4_angle_jitter_deg=12.0,
)

